# 🔍 Lab 04 — Convolutional Neural Networks

**DL2026 · Practical Session 4 · Slide set 04**

In Lab 02 you built a network that read handwritten digits by **flattening** every 28 × 28 image into a
list of 784 numbers. It worked — about 95 % — but it threw away the one thing a picture has that a list
does not: **neighbouring pixels belong together**. This week you build the layer that keeps that
information, the **convolution**, and the network made of it, the **Convolutional Neural Network (CNN)**.

> 🔑 **The through-line of this whole lab is one small neuron, slid over the image.**
> A convolution kernel is a neuron with a handful of weights (say 3 × 3 = 9) that looks at one small patch,
> then moves to the next patch, and the next, *re-using the same weights everywhere*. Every new word this
> week — **kernel size, padding, stride, channels, pooling, receptive field** — is a question about *how*
> that one neuron slides and *what it sees*. Nothing else is new: the loss, the gradient and the update are
> exactly what you wrote in Lab 02 and PyTorch-ified in Lab 03.

> 💡 **This is a hands-on notebook.** Reading it is not enough. Run every code cell (`Shift + Enter`),
> change the numbers, break things, and do the ✍️ exercises before opening the solutions.

**Rules of the game this week:** every layer is first computed **by hand** (a few lines of loops on a tiny
image), and only then handed to PyTorch — and we always check that the two agree with `torch.allclose`.
When a `Conv2d` refuses to give you the shape you expected, you will know *exactly* why.

## 🎯 Learning Outcomes

By the end of this lab you should be able to:

| # | You will be able to… | Section |
|:--|:---------------------|:--------|
| 1 | Say what an image is as a tensor (`N, C, H, W`), and **show experimentally** why a flattened MLP does not know it is looking at a picture | [§1](#s1) |
| 2 | Compute a 2-D **convolution by hand**, verify it with `F.conv2d`, and explain **weight sharing** | [§2](#s2) |
| 3 | Predict the output size of any convolution from **kernel size, padding, stride** (and dilation) with one formula | [§3](#s3) |
| 4 | Read the shape of a `Conv2d` weight tensor and **count its parameters** — and explain **channels** | [§4](#s4) |
| 5 | Explain what **ReLU, max/average pooling, flatten, dropout, batch norm** each do inside a CNN, and where they go | [§5](#s5) |
| 6 | Assemble a CNN as an `nn.Module`, **trace the shapes** through it, and compute its **receptive field** | [§6](#s6) |
| 7 | Train and evaluate a CNN on MNIST and compare it with the MLP — including on **shifted digits** | [§7](#s7) |
| 8 | **Look inside** a trained CNN: learned kernels, feature maps, most-activating images | [§8](#s8) |
| 9 | Run **controlled experiments** on kernel size, padding, pooling vs stride, width, dropout and batch norm | [§9](#s9) |

<a id="toc"></a>
## 📑 Table of Contents

| Section | Topic | What you practise |
|:--|:--|:--|
| [0](#s0) | **Setup & helpers** | environment check, device, plotting helpers |
| [1](#s1) | **Images as tensors, and the blind MLP** | MNIST in PyTorch, `(N, C, H, W)`, the training loop, shifted and shuffled pixels |
| [2](#s2) | **The convolution, by hand** | sliding a kernel, `F.conv2d`, edge detectors, weight sharing and equivariance |
| [3](#s3) | **Kernel size, padding, stride** | the output-size formula, `"same"` padding, downsampling, an interactive explorer |
| [4](#s4) | **Channels** | many kernels → many feature maps, the 4-D weight tensor, parameter counting |
| [5](#s5) | **The other layers** | ReLU, max/avg pooling, flatten + linear, dropout, batch norm, global average pooling |
| [6](#s6) | **Assembling a CNN** | `nn.Module`, shape tracing with hooks, parameter budget, receptive field |
| [7](#s7) | **Training on MNIST** | curves, test accuracy, shift robustness, confusion matrix, mistakes |
| [8](#s8) | **Looking inside** | learned kernels, feature maps layer by layer, most-activating images |
| [9](#s9) | **Experiments** | a configurable CNN factory and a results table you fill in |
| [10](#s10) | **Self-check quiz** | ten questions, instant feedback |
| [11](#s11) | **Summary & cheat sheet** | one page to keep |

⏱️ *Sections 1–6 are the concepts (about two hours). Sections 7–9 train networks — a GPU makes them
faster (Colab: `Runtime ▸ Change runtime type ▸ T4 GPU`) but the whole notebook runs on a CPU too.*

---
<a id="s0"></a>
# 0 · Setup & Helpers

[⬆ back to TOC](#toc)

Run the next three cells first. They check your environment, pick a device, and define the few helpers
this notebook reuses. Everything we need is pre-installed in Colab.

In [ ]:
import sys, platform

print("Python :", sys.version.split()[0], "on", platform.system())
print("Colab  :", "yes" if "google.colab" in sys.modules else "no (local Jupyter)")
print()

# Anything reported as MISSING can be installed with:
#   %pip install torch torchvision matplotlib numpy ipywidgets
for name in ["torch", "torchvision", "numpy", "matplotlib", "ipywidgets"]:
    try:
        module = __import__(name)
        print(f"{name:<12}: {getattr(module, '__version__', 'unknown')}")
    except ImportError:
        print(f"{name:<12}: MISSING")

### 🧪 Demo — Device, seed, and the helpers we reuse

* `DEVICE` — the GPU if there is one, otherwise the CPU. Every model and batch is moved there with `.to(DEVICE)`.
* `show_images(images, titles, ...)` — plot a list of 2-D tensors in a grid; with `annotate=True` it also
  writes every number into its cell. This is *the* plot of the week — we look at pixels, kernels and
  feature maps as pictures **and** as numbers.
* `conv_out_size(n, k, p, s, d)` — the one formula of [§3](#s3), as a function.
* `count_params(model)` — how many trainable numbers a model has.
* `render_mermaid(diagram)` — draw the diagrams of this notebook as images.

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})


def to_numpy(image):
    """Turn a tensor / array of any 2-D-ish shape into a float NumPy image (squeezes size-1 dims)."""
    if torch.is_tensor(image):
        image = image.detach().cpu()
    return np.asarray(image, dtype=np.float32).squeeze()


def show_images(images, titles=None, ncols=8, cmap="gray", size=1.7, annotate=False, fmt="{:.0f}",
                vmin=None, vmax=None, suptitle=None):
    """
    Plot a list (or a stacked tensor) of 2-D images in a grid.

    Args:
        images: Iterable of 2-D tensors/arrays, or a tensor of shape (n, H, W) / (n, 1, H, W).
        titles: Optional list of titles, one per image.
        ncols: Number of columns in the grid.
        cmap: Matplotlib colour map ('gray' for images, 'RdBu_r' for signed kernels / feature maps).
        size: Size of one cell in inches.
        annotate: If True, print every value in its pixel (use on small grids only).
        fmt: Format string for the annotations.
        vmin, vmax: Colour limits shared by all panels (None -> each panel autoscaled).
        suptitle: Optional title for the whole figure.
    """
    imgs = [to_numpy(im) for im in images]
    n = len(imgs)
    ncols = min(ncols, n)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(size * ncols, size * nrows + 0.3), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for k, (ax, im) in enumerate(zip(axes.flat, imgs)):
        lo = im.min() if vmin is None else vmin
        hi = im.max() if vmax is None else vmax
        ax.imshow(im, cmap=cmap, vmin=lo, vmax=hi)
        if titles is not None:
            ax.set_title(str(titles[k]), fontsize=9)
        if annotate:
            colormap = plt.get_cmap(cmap)
            for (i, j), v in np.ndenumerate(im):
                frac = 0.5 if hi == lo else float(np.clip((v - lo) / (hi - lo), 0, 1))
                r, g, b, _ = colormap(frac)                       # colour of this cell ...
                luminance = 0.299 * r + 0.587 * g + 0.114 * b     # ... and how bright it is
                ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=8,
                        color="white" if luminance < 0.5 else "black")
    if suptitle:
        fig.suptitle(suptitle, fontsize=10)
    plt.tight_layout()
    plt.show()
    plt.close(fig)


def conv_out_size(n, k, p=0, s=1, d=1):
    """Spatial output size of a convolution / pooling along one axis (the formula of section 3)."""
    return (n + 2 * p - d * (k - 1) - 1) // s + 1


def count_params(model):
    """Number of trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def render_mermaid(diagram):
    """Render a Mermaid diagram as an image; fall back to printing the source when offline."""
    import base64
    try:
        from IPython.display import Image, display
        payload = base64.urlsafe_b64encode(diagram.strip().encode("utf8")).decode("ascii")
        display(Image(url="https://mermaid.ink/img/" + payload))
    except Exception as error:
        print(f"[diagram not rendered: {error}]\n")
        print(diagram)


print("helpers ready")

---
<a id="s1"></a>
# 1 · Images as Tensors, and the Blind MLP

[⬆ back to TOC](#toc)

## 1.1 Loading MNIST with `torchvision`

In Lab 02 we fetched MNIST as a NumPy array. PyTorch's companion library **torchvision** ships the dataset
directly. `datasets.MNIST` downloads it once (about 12 MB) and returns `(image, label)` pairs; the
`transform` turns each image from a PIL picture with pixel values `0…255` into a **float tensor in
`[0, 1]`** of shape `(1, 28, 28)`.

That leading **1** is new. Read on.

In [ ]:
from torchvision import datasets, transforms

DATA_DIR = "./data"
to_tensor = transforms.ToTensor()      # PIL image (uint8, 0..255)  ->  float tensor (1, 28, 28) in [0, 1]

train_full = datasets.MNIST(DATA_DIR, train=True,  download=True, transform=to_tensor)
test_full  = datasets.MNIST(DATA_DIR, train=False, download=True, transform=to_tensor)

print(train_full)
print()
image, label = train_full[0]                       # one (image, label) pair
print("type(image) :", type(image).__name__)
print("image.shape :", tuple(image.shape), "   <- (channels, height, width)")
print("image.dtype :", image.dtype)
print("pixel range :", f"{image.min().item():.1f} .. {image.max().item():.1f}")
print("label       :", label)

### 🧪 Demo — A digit is a grid of numbers

Before any layer, look at the raw material. The centre of the first training image, printed as numbers
and drawn as a picture. **Every operation in this lab is arithmetic on grids like this one.**

In [ ]:
torch.set_printoptions(precision=1, linewidth=140)
print("the 10 x 10 centre of image 0 (label", label, "):")
print(image[0, 9:19, 9:19])
torch.set_printoptions(profile="default")

show_images([image[0, 9:19, 9:19]], titles=[f"centre patch, label {label}"], size=3.6,
            annotate=True, fmt="{:.1f}", vmin=0, vmax=1)

first_16 = [train_full[i][0] for i in range(16)]
show_images(first_16, titles=[train_full[i][1] for i in range(16)], ncols=8, size=1.3,
            suptitle="the first 16 training images, with labels")

## 1.2 The `(N, C, H, W)` convention

PyTorch stores a **batch of images** as a 4-D tensor:

| Axis | Name | MNIST | Colour photo |
|:--|:--|:--|:--|
| 0 | `N` — batch size | how many images at once | same |
| 1 | `C` — **channels** | 1 (grey) | 3 (red, green, blue) |
| 2 | `H` — height, in pixels | 28 | e.g. 224 |
| 3 | `W` — width, in pixels | 28 | e.g. 224 |

A single grey image is therefore `(1, 28, 28)`, and a batch of 128 of them is `(128, 1, 28, 28)`.
Every convolution layer we meet expects exactly this layout — **most shape errors this week are a missing
or extra axis**, so keep the table in view.

MNIST is small enough (70 000 × 784 bytes ≈ 55 MB) to keep entirely in memory as **one tensor**. That is
faster than converting images one at a time, and it lets us manipulate all images at once in [§1.4](#s14).
We split the official 60 000 training images into **55 000 for training** and **5 000 for validation**,
and keep the 10 000 test images sealed for final numbers only.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# The whole dataset as tensors: uint8 (N, 28, 28) -> float (N, 1, 28, 28) in [0, 1]
X_all = train_full.data.unsqueeze(1).float() / 255.0
y_all = train_full.targets
X_test = test_full.data.unsqueeze(1).float() / 255.0
y_test = test_full.targets

# train / validation split (fixed permutation, so everyone gets the same split)
perm = torch.randperm(len(X_all), generator=torch.Generator().manual_seed(SEED))
idx_train, idx_val = perm[:55000], perm[55000:]
X_train, y_train = X_all[idx_train], y_all[idx_train]
X_val,   y_val   = X_all[idx_val],   y_all[idx_val]

BATCH_SIZE = 128


def make_loader(X, y, shuffle=False, batch_size=BATCH_SIZE):
    """Wrap image and label tensors in a DataLoader."""
    return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=shuffle)


train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader   = make_loader(X_val,   y_val)
test_loader  = make_loader(X_test,  y_test)

print("X_train:", tuple(X_train.shape), "  y_train:", tuple(y_train.shape))
print("X_val  :", tuple(X_val.shape))
print("X_test :", tuple(X_test.shape))

images, labels = next(iter(train_loader))
print("\none mini-batch from the loader:", tuple(images.shape), "<- (N, C, H, W)", "  labels:", tuple(labels.shape))
print("class counts in the training split:", torch.bincount(y_train).tolist())

## 1.3 The training loop — the same five lines as always

Nothing here is new after Lab 03, so we write it **once** as two functions and reuse them for every model
in this notebook. The loss is **cross-entropy** on the raw outputs (*logits*) — the multi-class loss
from Lecture 03 — and the optimiser is **Adam**.

| Step | Code | Lab 02 equivalent |
|:--|:--|:--|
| forward | `logits = model(x)` | `z = X @ w + b`, layer by layer |
| loss | `F.cross_entropy(logits, y)` | the MSE you differentiated by hand |
| reset gradients | `optimizer.zero_grad()` | — |
| backward | `loss.backward()` | your backprop loop |
| step | `optimizer.step()` | `w -= eta * grad_w` |

`model.train()` / `model.eval()` switch layers such as dropout and batch norm ([§5.4](#s54)) between
their training and prediction behaviour; `torch.no_grad()` turns autograd off while evaluating.

In [ ]:
def evaluate(model, loader, device=DEVICE):
    """Average cross-entropy loss and accuracy of a model over a DataLoader."""
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss_sum += F.cross_entropy(logits, y, reduction="sum").item()
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += len(y)
    return loss_sum / total, correct / total


def train_model(model, train_loader, val_loader, epochs=3, lr=1e-3, device=DEVICE, verbose=True):
    """
    Train a classifier with Adam and cross-entropy; return the per-epoch history.

    Args:
        model: Any nn.Module mapping (N, 1, 28, 28) images to (N, 10) logits.
        train_loader, val_loader: DataLoaders of (images, labels).
        epochs: Number of passes over the training data.
        lr: Learning rate for Adam.
        device: Where to run.
        verbose: Print one line per epoch.

    Returns:
        dict with lists train_loss, train_acc, val_loss, val_acc and seconds.
    """
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "seconds": []}
    for epoch in range(1, epochs + 1):
        model.train()
        t0, loss_sum, correct, seen = time.time(), 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)                      # forward
            loss = F.cross_entropy(logits, y)      # loss
            optimizer.zero_grad()                  # reset gradients
            loss.backward()                        # backward
            optimizer.step()                       # update
            loss_sum += loss.item() * len(y)
            correct += (logits.argmax(dim=1) == y).sum().item()
            seen += len(y)
        val_loss, val_acc = evaluate(model, val_loader, device)
        history["train_loss"].append(loss_sum / seen)
        history["train_acc"].append(correct / seen)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["seconds"].append(time.time() - t0)
        if verbose:
            print(f"epoch {epoch:>2}/{epochs}  train loss {loss_sum / seen:.4f}  acc {correct / seen:.4f}  |  "
                  f"val loss {val_loss:.4f}  acc {val_acc:.4f}  |  {time.time() - t0:.1f}s")
    return history


def plot_history(histories, metric="val_acc", ax=None):
    """Plot one metric of several histories, given as a {name: history} dict."""
    if ax is None:
        _, ax = plt.subplots(figsize=(5.5, 3.6))
    for name, h in histories.items():
        ax.plot(range(1, len(h[metric]) + 1), h[metric], marker="o", label=name)
    ax.set_xlabel("epoch")
    ax.set_ylabel(metric.replace("_", " "))
    ax.grid(alpha=0.3)
    ax.legend()
    return ax


print("training helpers ready")

### 🧪 Demo — The Lab 02 network, in PyTorch: our baseline

The same architecture you wrote in NumPy in Lab 02 (`784 → hidden → 10`), with 100 hidden units and ReLU.
`nn.Flatten` is the `reshape(-1, 784)` that throws the picture away — remember that line, we will come back
to it.

In [ ]:
class MLP(nn.Module):
    """Flatten the image, then two fully-connected layers."""

    def __init__(self, n_hidden=100):
        super().__init__()
        self.flatten = nn.Flatten()                  # (N, 1, 28, 28) -> (N, 784)
        self.fc1 = nn.Linear(28 * 28, n_hidden)
        self.fc2 = nn.Linear(n_hidden, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        return self.fc2(x)                          # logits, one per class


torch.manual_seed(SEED)
mlp = MLP()
print(mlp)
print("parameters:", f"{count_params(mlp):,}")

history_mlp = train_model(mlp, train_loader, val_loader, epochs=2)
test_loss_mlp, test_acc_mlp = evaluate(mlp, test_loader)
print(f"\nMLP test accuracy: {test_acc_mlp:.4f}")

<a id="s14"></a>
## 1.4 Two experiments that show the MLP does not know it is looking at a picture

The MLP's first layer is `nn.Linear(784, 100)`: pixel number 0 and pixel number 1 are just two of 784
inputs. Nothing in the model records that they are **next to each other**, or that pixel 0 and pixel 28
are vertical neighbours. Two experiments make this concrete.

**Experiment A — shift the digits.** Move every test image a few pixels to the right and down
(`torch.roll`; the digits have empty borders, so nothing important wraps around). A human would not
notice. What happens to the MLP?

In [ ]:
def shift_images(X, dx, dy):
    """Move every image dx pixels right and dy pixels down (wrapping around the empty border)."""
    return torch.roll(X, shifts=(dy, dx), dims=(2, 3))


X_test_shifted = shift_images(X_test, dx=3, dy=3)
show_images(list(X_test[:6]) + list(X_test_shifted[:6]), ncols=6, size=1.3,
            titles=["original"] * 6 + ["shifted 3 px"] * 6)

acc_orig = evaluate(mlp, test_loader)[1]
acc_shift = evaluate(mlp, make_loader(X_test_shifted, y_test))[1]
print(f"MLP accuracy  original digits : {acc_orig:.4f}")
print(f"MLP accuracy  shifted 3 px    : {acc_shift:.4f}")

**Experiment B — shuffle the pixels.** Apply one fixed random permutation to the 784 pixel positions of
**every** image (train, validation and test alike). The images become unrecognisable to us. To the MLP
they are *exactly as easy*: it never used the arrangement of the inputs in the first place, only their
values, and a permutation of the inputs is just a permutation of the first-layer weights.

In [ ]:
pixel_perm = torch.randperm(28 * 28, generator=torch.Generator().manual_seed(1))


def shuffle_pixels(X):
    """Apply the SAME fixed permutation of pixel positions to every image."""
    return X.flatten(1)[:, pixel_perm].view(-1, 1, 28, 28)


show_images(list(X_test[:6]) + list(shuffle_pixels(X_test[:6])), ncols=6, size=1.3,
            titles=[f"label {y}" for y in y_test[:6].tolist()] + ["shuffled"] * 6)

torch.manual_seed(SEED)
mlp_shuffled = MLP()
history_mlp_shuffled = train_model(mlp_shuffled,
                                   make_loader(shuffle_pixels(X_train), y_train, shuffle=True),
                                   make_loader(shuffle_pixels(X_val), y_val), epochs=2)
acc_shuffled = evaluate(mlp_shuffled, make_loader(shuffle_pixels(X_test), y_test))[1]
print(f"\nMLP on real images     : {test_acc_mlp:.4f}")
print(f"MLP on shuffled pixels : {acc_shuffled:.4f}   <- essentially the same")

> 🔑 **What the two experiments say.** The MLP is a function of a *list* of 784 numbers. It is
> **not translation-robust** (Experiment A) because a shifted digit puts familiar values on unfamiliar
> inputs. It is **blind to spatial arrangement** (Experiment B) because it never used it. Both facts come
> from the same line: `nn.Flatten`.
>
> A convolutional layer is the opposite on both counts: it looks at **small neighbourhoods** (locality), and
> it uses **the same weights at every position** (weight sharing). At the end of the lab we repeat both
> experiments on a CNN.

### ✍️ Exercise 1 — Measure the blindness

1. **A shift curve.** Evaluate `mlp` on test images shifted by `dx = dy = s` for `s` in `0, 1, 2, …, 6`
   and plot accuracy against shift. Where does it fall below 50 %?
2. **Which digits break first?** For `s = 3`, compute the per-class accuracy (a loop over the ten labels,
   or `torch.bincount`). Which classes survive the shift best, and can you guess why?
3. **Flip, not shift.** Mirror the test images left–right (`torch.flip(X, dims=[3])`) and evaluate. Should
   a *good* digit reader be robust to this? (Think about 2 and 5, or 6 and 9 for a vertical flip.)

In [ ]:
# ✍️ YOUR CODE HERE
# 1) accuracies for shifts 0..6, then plt.plot
# 2) per-class accuracy at shift 3
# 3) left-right flip: torch.flip(X_test, dims=[3])

<details>
<summary>✅ <b>Show solution</b></summary>

```python
# 1) shift curve
shifts = list(range(7))
accs = [evaluate(mlp, make_loader(shift_images(X_test, s, s), y_test))[1] for s in shifts]
plt.figure(figsize=(5, 3.3))
plt.plot(shifts, accs, marker="o")
plt.xlabel("shift (pixels, right and down)"); plt.ylabel("test accuracy"); plt.grid(alpha=0.3)
plt.show()
for s, a in zip(shifts, accs):
    print(f"shift {s}: {a:.3f}")

# 2) per-class accuracy at shift 3
mlp.eval()
with torch.no_grad():
    pred = torch.cat([mlp(x.to(DEVICE)).argmax(1).cpu()
                      for x, _ in make_loader(shift_images(X_test, 3, 3), y_test)])
for c in range(10):
    mask = y_test == c
    print(f"class {c}: {(pred[mask] == c).float().mean():.3f}")
# Which classes survive is hard to predict (in our run 0 and 1 collapsed completely while 2, 3, 5, 7 kept
# about 30 %) - and that is the point: the MLP has no notion of position, so its failure pattern is arbitrary.

# 3) mirror
acc_flip = evaluate(mlp, make_loader(torch.flip(X_test, dims=[3]), y_test))[1]
print(f"left-right flipped: {acc_flip:.3f}")
# Some robustness to mirroring is NOT desirable: a mirrored 2 is not a 2 in any handwriting.
# Translation robustness is desirable; mirror robustness is a design decision. Keep this in mind
# when you meet data augmentation.
```

</details>

---
<a id="s2"></a>
# 2 · The Convolution, by Hand

[⬆ back to TOC](#toc)

## 2.1 One small neuron, slid over the image

Take a tiny grid of weights — a **kernel** (also called a *filter*), say 3 × 3. Lay it over the top-left
3 × 3 patch of the image. Multiply each weight by the pixel underneath, add the nine products (plus a bias),
and write the single result into the top-left cell of a new grid. Slide the kernel one pixel to the right
and repeat. When you reach the right edge, go one row down and start again.

$$
\text{out}[i, j] \;=\; b \;+\; \sum_{u=0}^{k-1}\sum_{v=0}^{k-1} K[u, v]\cdot X[i+u,\; j+v]
$$

That is the whole operation. Compare it with Lab 02: one output cell is **exactly one neuron**
($z = w^\top x + b$) whose inputs are the $k^2$ pixels under the kernel. The convolution is that neuron
**copied to every position with the same weights**. The new grid it produces is called a **feature map**.

> 📝 **A naming detail.** Mathematicians call this operation *cross-correlation*; a true *convolution* flips
> the kernel first. Deep-learning libraries, papers and this course all say "convolution" and mean the
> unflipped version — since the kernel is learned anyway, the flip changes nothing.

In [ ]:
conv_diagram = """
flowchart LR
    X["input image<br/>H × W"] --> P["take the k × k patch<br/>at position (i, j)"]
    K["kernel K<br/>k × k weights + bias"] --> M["multiply element-wise,<br/>sum, add bias"]
    P --> M
    M --> O["one number ->  out[i, j]"]
    O -->|"slide to the next position"| P
    O --> FM["feature map<br/>(H-k+1) × (W-k+1)"]
"""
render_mermaid(conv_diagram)

### 🧪 Demo — Convolution in four lines of loops

A 6 × 6 image that is dark on the left and bright on the right, and a kernel that subtracts the left column
of each patch from the right one. Watch what the output picks out.

In [ ]:
image = torch.zeros(6, 6)
image[:, 3:] = 1.0                                   # dark left half, bright right half

kernel = torch.tensor([[-1.0, 0.0, 1.0],
                       [-1.0, 0.0, 1.0],
                       [-1.0, 0.0, 1.0]])             # "right column minus left column"


def conv2d_by_hand(image, kernel, bias=0.0):
    """2-D convolution (cross-correlation) of one channel, no padding, stride 1, with loops."""
    H, W = image.shape
    k = kernel.shape[0]
    out = torch.zeros(H - k + 1, W - k + 1)
    for i in range(H - k + 1):
        for j in range(W - k + 1):
            patch = image[i:i + k, j:j + k]           # the k x k window under the kernel
            out[i, j] = (patch * kernel).sum() + bias # multiply element-wise, add up
    return out


out = conv2d_by_hand(image, kernel)
print("image :", tuple(image.shape), " kernel:", tuple(kernel.shape), " ->  output:", tuple(out.shape))
show_images([image, kernel, out], titles=["image 6×6", "kernel 3×3", "output 4×4"], size=2.6,
            annotate=True, cmap="RdBu_r", vmin=-3, vmax=3)

Two things to read off the picture:

* The output is **smaller** (6 → 4): the kernel's centre can only visit positions where the whole 3 × 3
  window fits inside the image. $6 - 3 + 1 = 4$. [§3](#s3) fixes this with *padding*.
* The output is **3 exactly where the edge is** and 0 everywhere the patch is uniform. The kernel is an
  **edge detector** for dark-to-bright vertical edges. Flip its sign and it detects bright-to-dark.

Here are the first three windows in slow motion:

In [ ]:
for j in range(3):
    patch = image[0:3, j:j + 3]
    print(f"output position (0, {j}):   patch =\n{patch.int().numpy()}")
    print(f"   patch * kernel = \n{(patch * kernel).int().numpy()}")
    print(f"   sum = {(patch * kernel).sum().item():.0f}\n")

### 🧪 Demo — The same thing in PyTorch: `F.conv2d`

PyTorch's `F.conv2d(input, weight)` expects the **4-D layout** from [§1.2](#s1):

| Tensor | Shape | Ours |
|:--|:--|:--|
| `input` | `(N, C_in, H, W)` | `(1, 1, 6, 6)` — one image, one channel |
| `weight` | `(C_out, C_in, k, k)` | `(1, 1, 3, 3)` — one kernel, for one input channel |
| output | `(N, C_out, H_out, W_out)` | `(1, 1, 4, 4)` |

`image[None, None]` adds the two missing axes (the same as `image.unsqueeze(0).unsqueeze(0)`).
Channels are the subject of [§4](#s4); for now both are 1.

In [ ]:
out_torch = F.conv2d(image[None, None], kernel[None, None])     # (1,1,6,6) conv (1,1,3,3) -> (1,1,4,4)
print("F.conv2d output shape:", tuple(out_torch.shape))
print("identical to our loops?", torch.allclose(out_torch[0, 0], out))

## 2.2 Hand-designed kernels on a real digit

Before 2012, computer-vision engineers designed kernels like these **by hand**, for decades. A CNN
**learns** its kernels from data instead — but the learned ones often end up looking very similar to these
four, which is why it pays to know them.

| Kernel | What it responds to |
|:--|:--|
| vertical edges (Sobel-x) | a change of brightness from left to right |
| horizontal edges (Sobel-y) | a change of brightness from top to bottom |
| blur (box) | the average of the neighbourhood — smooths noise, loses detail |
| sharpen | the centre pixel *minus* its neighbours — exaggerates detail |

In [ ]:
digit = X_test[0, 0]                                    # a 28 x 28 tensor; the label is y_test[0]
print("this digit is a", y_test[0].item())

HAND_KERNELS = {
    "vertical edges":   torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]),
    "horizontal edges": torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]),
    "blur":             torch.ones(3, 3) / 9.0,
    "sharpen":          torch.tensor([[0., -1., 0.], [-1., 5., -1.], [0., -1., 0.]]),
}

show_images(list(HAND_KERNELS.values()), titles=list(HAND_KERNELS.keys()), size=2.2,
            annotate=True, fmt="{:.1f}", cmap="RdBu_r", vmin=-2, vmax=2, suptitle="the kernels")

feature_maps = [F.conv2d(digit[None, None], k[None, None])[0, 0] for k in HAND_KERNELS.values()]
show_images([digit] + feature_maps, titles=["input 28×28"] + [f"{n} {tuple(m.shape)}" for n, m in
            zip(HAND_KERNELS, feature_maps)], size=2.2, cmap="RdBu_r", suptitle="the feature maps (26×26)")

Read the maps: the **vertical-edge** kernel lights up (red or blue depending on direction) on the
*vertical* strokes and is silent on the horizontal bar; the **horizontal-edge** kernel does the opposite.
Neither cares *where* in the image the stroke is — an edge in the corner produces the same response as an
edge in the centre. That is the property the MLP lacked.

## 2.3 Weight sharing, and what it buys

| | Dense neuron (Lab 02) | Convolution kernel |
|:--|:--|:--|
| inputs per output | all 784 pixels | one $k \times k$ patch (9 pixels) |
| parameters | 784 weights + 1 bias = **785** | 9 weights + 1 bias = **10** |
| outputs | 1 number | a whole feature map (26 × 26 = 676 numbers) |
| a pattern in a new place | must be re-learned there | detected automatically |

The last row has a name: convolution is **translation-equivariant** — *shift the input, and the output
shifts by the same amount*. Let us verify it numerically, and show that a linear layer does not have it.

In [ ]:
sobel = HAND_KERNELS["vertical edges"][None, None]
x = digit[None, None]

# convolution: shift-then-convolve  ==  convolve-then-shift
conv_of_shifted = F.conv2d(shift_images(x, 3, 0), sobel, padding=1)
shifted_conv    = shift_images(F.conv2d(x, sobel, padding=1), 3, 0)
print("conv:   max |conv(shift(x)) - shift(conv(x))| =", f"{(conv_of_shifted - shifted_conv).abs().max():.4f}")

# a linear layer (random weights): shift-then-apply  !=  apply-then-shift
torch.manual_seed(SEED)
linear = nn.Linear(784, 784)
lin_of_shifted = linear(shift_images(x, 3, 0).flatten(1)).view(1, 1, 28, 28)
shifted_lin    = shift_images(linear(x.flatten(1)).view(1, 1, 28, 28), 3, 0)
print("linear: max |lin(shift(x))  - shift(lin(x))|  =", f"{(lin_of_shifted - shifted_lin).abs().max():.4f}")

show_images([shift_images(x, 3, 0)[0, 0], conv_of_shifted[0, 0], shifted_conv[0, 0]], size=2.2, cmap="RdBu_r",
            titles=["x shifted 3 px", "conv(shift(x))", "shift(conv(x))"])

> 🔑 **Key idea.** A convolution layer is a *dense layer with two constraints*: each output looks only at a
> **local** patch, and all outputs **share** the same weights. The constraints cost nothing on images
> (patterns *are* local and *do* recur) and buy three things: far fewer parameters, detectors that work
> anywhere in the image, and a network that can be trained with far less data.

### ✍️ Exercise 2 — Kernels by hand

1. **Bias and a check.** `conv2d_by_hand` already accepts a `bias`. Verify it against
   `F.conv2d(..., bias=torch.tensor([b]))` for `b = 0.5` on the digit (use `torch.allclose`).
2. **Identity and shift.** Write the 3 × 3 kernel that reproduces the input exactly (apart from the lost
   border), and another that *moves* the image one pixel to the left. Check both on `digit`.
3. **A diagonal detector.** Design a 3 × 3 kernel that responds to a stroke going from bottom-left to
   top-right (like the long stroke of a 7). Apply it to `digit` and to a few other test digits and look at
   where it fires.
4. **Bigger is blurrier.** Apply a 3 × 3, a 7 × 7 and an 11 × 11 box blur to `digit`. Besides the blur,
   what happens to the output size?

In [ ]:
# ✍️ YOUR CODE HERE
# 1) bias check vs F.conv2d(..., bias=torch.tensor([0.5]))
# 2) identity kernel; "move left" kernel
# 3) a diagonal (/) detector
# 4) box blurs of size 3, 7, 11

<details>
<summary>✅ <b>Show solution</b></summary>

```python
# 1) bias
k = HAND_KERNELS["vertical edges"]
mine = conv2d_by_hand(digit, k, bias=0.5)
ref = F.conv2d(digit[None, None], k[None, None], bias=torch.tensor([0.5]))[0, 0]
print("bias check:", torch.allclose(mine, ref, atol=1e-5))

# 2) identity: a 1 in the centre. "Move left": the 1 one step to the RIGHT of the centre
#    (the output at (i, j) copies the input at (i, j+1), so the picture moves left).
identity = torch.tensor([[0., 0., 0.], [0., 1., 0.], [0., 0., 0.]])
move_left = torch.tensor([[0., 0., 0.], [0., 0., 1.], [0., 0., 0.]])
print("identity reproduces the interior:", torch.allclose(conv2d_by_hand(digit, identity), digit[1:-1, 1:-1]))
show_images([digit, conv2d_by_hand(digit, identity), conv2d_by_hand(digit, move_left)],
            titles=["input", "identity", "moved left"], size=2.2)

# 3) a "/" detector: positive along the anti-diagonal, negative elsewhere
diag = torch.tensor([[-1., -1., 2.], [-1., 2., -1.], [2., -1., -1.]])
maps = [F.conv2d(X_test[i, 0][None, None], diag[None, None])[0, 0] for i in range(6)]
show_images(list(X_test[:6, 0]) + maps, ncols=6, size=1.6, cmap="RdBu_r",
            titles=[f"label {y}" for y in y_test[:6].tolist()] + ["/ detector"] * 6)

# 4) box blurs: output size 28 - k + 1 -> 26, 22, 18
for k in (3, 7, 11):
    box = torch.ones(k, k) / k**2
    print(f"box {k:>2}x{k:<2}: output {tuple(conv2d_by_hand(digit, box).shape)}")
```

The output shrinks by $k-1$ pixels in each direction. That is the first reason for **padding**, next.

</details>

---
<a id="s3"></a>
# 3 · Kernel Size, Padding, Stride

[⬆ back to TOC](#toc)

Three numbers decide *how the kernel slides*. Together with the input size they fix the output size, and
nothing else does. Learn the formula and you will never be surprised by a shape again:

$$
n_{\text{out}} \;=\; \left\lfloor \frac{n_{\text{in}} + 2p - k}{s} \right\rfloor + 1
$$

| Symbol | `nn.Conv2d` argument | Meaning |
|:--|:--|:--|
| $k$ | `kernel_size` | side of the kernel: how many pixels one output looks at |
| $p$ | `padding` | rows/columns of zeros added around the input **on each side** |
| $s$ | `stride` | how many pixels the kernel jumps between two output positions |

(The formula applies per axis; height and width can have different $k$, $p$, $s$ but almost never do.)

## 3.1 Kernel size $k$

A larger kernel sees a bigger patch, so it can detect bigger patterns in one step — at the price of
$k^2$ parameters instead of 9, and a smaller output. Odd sizes (3, 5, 7) are used almost exclusively,
because an odd kernel has a **centre pixel**, which makes "the output at (i, j) describes the input around
(i, j)" literally true.

In [ ]:
for k in (3, 5, 7):
    conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=k)     # random kernel, p = 0, s = 1
    out = conv(digit[None, None])
    print(f"k = {k}:  28x28  ->  {tuple(out.shape[2:])}   ({count_params(conv)} parameters)   "
          f"formula: {conv_out_size(28, k)}")

## 3.2 Padding $p$

Two problems with $p = 0$: the output shrinks with every layer (a 28 × 28 image survives only thirteen
3 × 3 layers), and **border pixels are seen far less often** than central ones — a corner pixel is inside
exactly one 3 × 3 window; a central pixel is inside nine.

The fix is to surround the input with a frame of $p$ zeros. With $p = (k - 1)/2$ the output has **the same
size as the input** — so common that PyTorch accepts `padding="same"` as a shortcut (for stride 1).

In [ ]:
padded = F.pad(digit, pad=(2, 2, 2, 2))            # (left, right, top, bottom) zeros -> 32 x 32
print("padded shape:", tuple(padded.shape))

fig, axes = plt.subplots(1, 2, figsize=(6.2, 3.1))
axes[0].imshow(digit, cmap="gray"); axes[0].set_title("input 28×28"); axes[0].axis("off")
axes[1].imshow(padded, cmap="gray"); axes[1].set_title("padded with p = 2  ->  32×32"); axes[1].axis("off")
axes[1].add_patch(plt.Rectangle((1.5, 1.5), 28, 28, fill=False, edgecolor="orange", lw=1.5))
plt.tight_layout(); plt.show()

for k in (3, 5, 7):
    p = (k - 1) // 2
    out = nn.Conv2d(1, 1, kernel_size=k, padding=p)(digit[None, None])
    print(f"k = {k}, p = {p}:  28x28  ->  {tuple(out.shape[2:])}    formula: {conv_out_size(28, k, p)}")

out_same = nn.Conv2d(1, 1, kernel_size=5, padding="same")(digit[None, None])
print('k = 5, padding="same":  ->', tuple(out_same.shape[2:]))

## 3.3 Stride $s$

With stride 1 the kernel visits every position. With stride 2 it visits every *second* position in each
direction, so the output has roughly **half** the height and width — a quarter of the values. Stride is the
cheapest way to **downsample**: the network keeps a coarser summary and the following layers run four times
faster. (The other way, *pooling*, comes in [§5.2](#s52).)

In [ ]:
torch.manual_seed(SEED)
conv3 = nn.Conv2d(1, 1, kernel_size=3, padding=1)     # same random kernel for a fair comparison
outs, titles = [digit], ["input 28×28"]
for s in (1, 2, 4):
    out = F.conv2d(digit[None, None], conv3.weight, conv3.bias, stride=s, padding=1)[0, 0]
    print(f"s = {s}:  28x28  ->  {tuple(out.shape)}    formula: {conv_out_size(28, 3, 1, s)}")
    outs.append(out); titles.append(f"stride {s}: {tuple(out.shape)}")
show_images(outs, titles=titles, size=2.2, cmap="RdBu_r")

The stride-2 map shows the *same* pattern as the stride-1 map, at half the resolution. Note that with
$n = 28, k = 3, p = 1, s = 2$ the formula gives $\lfloor 27/2 \rfloor + 1 = 14$ — the floor matters: when
$(n + 2p - k)$ is not a multiple of $s$, the last partial step is simply **dropped**.

## 3.4 One formula, checked against PyTorch — and dilation

`conv_out_size` from the helpers implements the formula with one extra parameter, **dilation** $d$: the
kernel's taps are spread $d$ pixels apart, so a 3 × 3 kernel with $d = 2$ covers a 5 × 5 area with only
9 weights. The effective kernel size is $d(k-1) + 1$, and the formula becomes

$$
n_{\text{out}} = \left\lfloor \frac{n_{\text{in}} + 2p - d(k-1) - 1}{s} \right\rfloor + 1 .
$$

Dilation is rare in classification networks and common in segmentation; you only need to recognise it.

In [ ]:
configs = [  # (k, p, s, d)
    (3, 0, 1, 1), (3, 1, 1, 1), (5, 2, 1, 1), (7, 0, 1, 1),
    (3, 1, 2, 1), (5, 2, 2, 1), (4, 0, 2, 1), (2, 0, 2, 1),
    (3, 0, 1, 2), (3, 2, 1, 2), (3, 1, 3, 1),
]
print(f"{'k':>3} {'p':>3} {'s':>3} {'d':>3} | {'formula':>8} | {'PyTorch':>8}")
print("-" * 44)
for k, p, s, d in configs:
    conv = nn.Conv2d(1, 1, kernel_size=k, padding=p, stride=s, dilation=d)
    actual = conv(digit[None, None]).shape[-1]
    predicted = conv_out_size(28, k, p, s, d)
    assert actual == predicted, (k, p, s, d)
    print(f"{k:>3} {p:>3} {s:>3} {d:>3} | {predicted:>8} | {actual:>8}")
print("\nall predictions match")

### 🎛️ Interactive — Kernel / padding / stride explorer

Move the sliders and watch three things: the **padded input** with the first two kernel windows drawn on it
(the gap between them is the stride), the **output** and its **size**. If widgets are unavailable the cell
prints a small static sweep instead.

In [ ]:
def explore_conv(k=3, p=0, s=1):
    """Draw the padded input, the first two kernel windows and the resulting feature map."""
    n_out = conv_out_size(28, k, p, s)
    if n_out < 1:
        print(f"k={k}, p={p}, s={s}: the kernel ({k}x{k}) is larger than the padded input ({28 + 2 * p}) -> no output")
        return
    weight = HAND_KERNELS["vertical edges"] if k == 3 else torch.ones(k, k) / k**2
    out = F.conv2d(digit[None, None], weight[None, None], padding=p, stride=s)[0, 0]
    padded = F.pad(digit, (p, p, p, p))

    fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.6))
    axes[0].imshow(padded, cmap="gray")
    axes[0].add_patch(plt.Rectangle((-0.5, -0.5), k, k, fill=False, edgecolor="orange", lw=2))
    axes[0].add_patch(plt.Rectangle((s - 0.5, -0.5), k, k, fill=False, edgecolor="deepskyblue", lw=2, ls="--"))
    axes[0].set_title(f"padded input {padded.shape[0]}×{padded.shape[1]}  (p = {p})")
    axes[1].imshow(out, cmap="RdBu_r")
    axes[1].set_title(f"output {n_out}×{n_out}")
    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()
    print(f"n_out = floor((28 + 2*{p} - {k}) / {s}) + 1 = floor({28 + 2 * p - k} / {s}) + 1 = {n_out}")


try:
    from ipywidgets import interact, IntSlider
    interact(explore_conv,
             k=IntSlider(3, min=1, max=11, step=2, description="kernel k"),
             p=IntSlider(0, min=0, max=5, description="padding p"),
             s=IntSlider(1, min=1, max=4, description="stride s"))
except ImportError:
    print("[ipywidgets not available - showing a static sweep]")
    for k, p, s in [(3, 0, 1), (3, 1, 1), (5, 0, 2), (7, 3, 2)]:
        explore_conv(k, p, s)

### ✍️ Exercise 3 — Shapes on paper first, then in code

1. **Predict, then verify.** For a 28 × 28 input, write down the output size for each of these *before*
   running anything, then check with `conv_out_size` **and** an actual `nn.Conv2d`:
   `(k=5, p=0, s=1)`, `(k=5, p=2, s=2)`, `(k=3, p=1, s=3)`, `(k=1, p=0, s=1)`, `(k=7, p=0, s=3)`.
2. **All roads to 14 × 14.** Loop over `k ∈ {3, 5, 7}`, `p ∈ {0, 1, 2, 3}`, `s ∈ {1, 2}` and list every
   combination whose output is exactly 14 × 14. Which single combination is the "usual" one?
3. **Two layers.** A `(k=3, p=1, s=1)` convolution followed by a `(k=3, p=0, s=2)` convolution: what is the
   final size? Compose `conv_out_size` twice, then verify with `nn.Sequential`.
4. **Pixels that are never seen.** With `k=3, p=0, s=2` on 28 × 28, the floor drops a partial step. Which
   input row/column is never inside any window? Verify by convolving an image that is 1 only in that last
   row and column with a kernel of ones — the output should be all zeros.

In [ ]:
# ✍️ YOUR CODE HERE
# 1) predictions vs conv_out_size vs nn.Conv2d
# 2) the loop that finds every (k, p, s) with output 14
# 3) conv_out_size(conv_out_size(28, 3, 1, 1), 3, 0, 2); verify with nn.Sequential
# 4) an image with ones only in the last row and column, k=3, s=2, kernel of ones

<details>
<summary>✅ <b>Show solution</b></summary>

```python
# 1) predictions: 24, 14, 10, 28, 8
for k, p, s in [(5, 0, 1), (5, 2, 2), (3, 1, 3), (1, 0, 1), (7, 0, 3)]:
    actual = nn.Conv2d(1, 1, k, padding=p, stride=s)(digit[None, None]).shape[-1]
    print(f"k={k} p={p} s={s}: formula {conv_out_size(28, k, p, s):>2}   PyTorch {actual:>2}")

# 2) every way to reach 14
hits = [(k, p, s) for k in (3, 5, 7) for p in range(4) for s in (1, 2) if conv_out_size(28, k, p, s) == 14]
print("output 14x14 for (k, p, s) in:", hits)
# The usual one is (3, 1, 2): "same" padding with stride 2 - the standard downsampling convolution.
# With s = 1 nothing reaches 14: you cannot halve an image without a stride (or a pooling layer).

# 3) two layers: 28 -> 28 -> 13
print("two layers:", conv_out_size(conv_out_size(28, 3, 1, 1), 3, 0, 2))
two = nn.Sequential(nn.Conv2d(1, 4, 3, padding=1), nn.Conv2d(4, 4, 3, stride=2))
print("PyTorch   :", tuple(two(digit[None, None]).shape))

# 4) with n=28, k=3, s=2 the windows cover rows 0..26 (0-2, 2-4, ..., 24-26); row 27 is never seen.
probe = torch.zeros(28, 28); probe[27, :] = 1; probe[:, 27] = 1
out = F.conv2d(probe[None, None], torch.ones(1, 1, 3, 3), stride=2)
print("output max when only the last row/column is lit:", out.max().item(), " (shape", tuple(out.shape[2:]), ")")
# This is why people prefer even inputs with even strides (and padding), or (k=2, s=2) pooling: no pixel is wasted.
```

</details>

---
<a id="s4"></a>
# 4 · Channels: Many Kernels, Many Feature Maps

[⬆ back to TOC](#toc)

## 4.1 One kernel is one detector; a layer has many

A single kernel detects **one** kind of pattern. A useful layer needs several — vertical edges *and*
horizontal edges *and* blobs *and* … — so a convolution layer holds **`out_channels` kernels**, each
producing its own feature map. The feature maps are stacked along the channel axis: the output of a layer
with 8 kernels applied to a `(N, 1, 28, 28)` batch is `(N, 8, 28, 28)`.

The next layer then receives an input with **8 channels**. Its kernels must look at all of them at once,
so each kernel is now a **3-D block** of shape `(in_channels, k, k)`: it multiplies a $k \times k$ patch in
*every* input channel by its own $k \times k$ slice of weights, and sums **everything** into one number.

$$
\text{out}[c_{\text{out}}, i, j] \;=\; b_{c_{\text{out}}} \;+\; \sum_{c_{\text{in}}}\;\sum_{u}\sum_{v}
W[c_{\text{out}}, c_{\text{in}}, u, v]\cdot X[c_{\text{in}},\, i+u,\, j+v]
$$

That is why a `Conv2d` weight is **4-D**:

| `conv.weight.shape` | `(out_channels, in_channels, k, k)` |
|:--|:--|
| `conv.bias.shape` | `(out_channels,)` — one bias per kernel |
| parameters | $\;C_{\text{out}} \cdot C_{\text{in}} \cdot k \cdot k \;+\; C_{\text{out}}$ |

Notice what is **not** in that count: the image size. A convolution layer has the same number of
parameters for a 28 × 28 digit and a 4000 × 3000 photo.

In [ ]:
channels_diagram = """
flowchart LR
    X["input<br/>(1, 28, 28)<br/>1 channel"] -->|"8 kernels<br/>each (1, 3, 3)"| A["feature maps<br/>(8, 28, 28)<br/>8 channels"]
    A -->|"16 kernels<br/>each (8, 3, 3)"| B["feature maps<br/>(16, 28, 28)<br/>16 channels"]
    W1["weight (8, 1, 3, 3)<br/>bias (8,)<br/>= 80 parameters"] -.- A
    W2["weight (16, 8, 3, 3)<br/>bias (16,)<br/>= 1168 parameters"] -.- B
"""
render_mermaid(channels_diagram)

### 🧪 Demo — A layer of eight kernels

`nn.Conv2d` creates the weight tensor (randomly initialised, like `nn.Linear`) and the bias for us.

In [ ]:
torch.manual_seed(SEED)
conv1 = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1)

print(conv1)
print("weight shape :", tuple(conv1.weight.shape), "  <- (out_channels, in_channels, k, k)")
print("bias shape   :", tuple(conv1.bias.shape))
print("parameters   :", count_params(conv1), "=", "8 * 1 * 3 * 3 + 8 =", 8 * 1 * 3 * 3 + 8)

batch = X_test[:64]                                  # (64, 1, 28, 28)
maps = conv1(batch)
print("\ninput batch  :", tuple(batch.shape))
print("output       :", tuple(maps.shape), "  <- one feature map per kernel, for every image")

show_images(conv1.weight[:, 0], titles=[f"kernel {i}" for i in range(8)], size=1.5, cmap="RdBu_r",
            annotate=True, fmt="{:.1f}", suptitle="the 8 random kernels (each is weight[i, 0])")
show_images([batch[0, 0]] + list(maps[0]), titles=["input"] + [f"map {i}" for i in range(8)], ncols=9,
            size=1.5, cmap="RdBu_r", suptitle="image 0 through the 8 kernels (random weights, before training)")

The random kernels already act like crude edge and blob detectors — random weights that sum to
something non-zero behave like a blurred copy, random weights with mixed signs behave like edge detectors.
Training will *sharpen* these into useful ones ([§8](#s8)).

### 🧪 Demo — The second layer: a kernel is a 3-D block

Feed the 8 maps into a layer with 16 kernels. Each kernel is `(8, 3, 3)`: 72 weights that look at a
3 × 3 patch **in all 8 maps at once**.

In [ ]:
torch.manual_seed(SEED)
conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, padding=1)
maps2 = conv2(maps)

print("conv2.weight :", tuple(conv2.weight.shape), " -> each of the 16 kernels is an (8, 3, 3) block")
print("input        :", tuple(maps.shape))
print("output       :", tuple(maps2.shape))
print("parameters   :", count_params(conv2), "=", "16 * 8 * 3 * 3 + 16 =", 16 * 8 * 3 * 3 + 16)

show_images(conv2.weight[0], titles=[f"kernel 0, slice {c}" for c in range(8)], size=1.5, cmap="RdBu_r",
            annotate=True, fmt="{:.1f}", suptitle="the 8 slices of ONE conv2 kernel - one per input channel")

# The four hand-made kernels of section 2 are simply a conv layer with out_channels = 4:
hand_weight = torch.stack(list(HAND_KERNELS.values()))[:, None]     # (4, 1, 3, 3)
four_maps = F.conv2d(digit[None, None], hand_weight, padding=1)
print("\nfour hand kernels at once:", tuple(hand_weight.shape), "->", tuple(four_maps.shape))

### 🧪 Demo — Colour images and the 1 × 1 convolution

Two more shapes worth seeing once. A **colour image** simply has `in_channels = 3`; every kernel of the first
layer is then a `(3, k, k)` block, and the parameter count grows by 3, not by the number of pixels.
A **1 × 1 convolution** has no spatial extent at all: it only *mixes channels* at each position (a tiny
linear layer applied to every pixel). It is the standard way to change the number of channels cheaply.

In [ ]:
rgb_batch = torch.rand(2, 3, 64, 48)                            # 2 colour images, 64 x 48
conv_rgb = nn.Conv2d(3, 8, kernel_size=5, padding=2)
print("RGB input", tuple(rgb_batch.shape), "-> weight", tuple(conv_rgb.weight.shape),
      "->", tuple(conv_rgb(rgb_batch).shape), "  params:", count_params(conv_rgb), "= 8*3*25 + 8")

conv_1x1 = nn.Conv2d(16, 4, kernel_size=1)
print("1x1 conv  ", tuple(maps2.shape), "-> weight", tuple(conv_1x1.weight.shape),
      "->", tuple(conv_1x1(maps2).shape), "  params:", count_params(conv_1x1), "= 4*16*1 + 4")

### ✍️ Exercise 4 — Own the 4-D weight

1. **Count first.** Without running anything, write down the number of parameters of
   `nn.Conv2d(3, 64, 7)`, `nn.Conv2d(64, 64, 3)`, `nn.Conv2d(64, 128, 1)` and `nn.Conv2d(256, 512, 3)`.
   Then verify with `count_params`.
2. **Multi-channel convolution by hand.** Extend `conv2d_by_hand` to an input of shape `(C_in, H, W)` and
   a weight of shape `(C_out, C_in, k, k)` plus a bias of shape `(C_out,)`: loop over output channels, and
   for each one sum the single-channel convolutions over the input channels. Verify against `F.conv2d` on
   `x = torch.randn(2, 5, 5)`, `w = torch.randn(3, 2, 3, 3)`, `b = torch.randn(3)`.
3. **Size-independent.** Apply `conv1` to a `(1, 1, 28, 28)` batch and to a `(1, 1, 100, 100)` batch.
   Compare parameter counts and output shapes. Then do the same with `nn.Linear(28*28, 8)` — what goes wrong?
4. **Reading a weight.** In `conv2.weight[3, 5]`, what is 3 and what is 5? What does the slice
   `conv2.weight[:, 5]` contain, and what is its shape?

In [ ]:
# ✍️ YOUR CODE HERE
# 1) four parameter counts by hand, then count_params
# 2) def conv2d_multichannel_by_hand(x, weight, bias): ...
# 3) conv1 on 28x28 and on 100x100; nn.Linear on both
# 4) conv2.weight[3, 5] and conv2.weight[:, 5]

<details>
<summary>✅ <b>Show solution</b></summary>

```python
# 1) 3*64*49 + 64 = 9,472 ; 64*64*9 + 64 = 36,928 ; 64*128*1 + 128 = 8,320 ; 256*512*9 + 512 = 1,180,160
for c_in, c_out, k in [(3, 64, 7), (64, 64, 3), (64, 128, 1), (256, 512, 3)]:
    print(f"Conv2d({c_in}, {c_out}, {k}): by hand {c_out * c_in * k * k + c_out:>9,}   torch {count_params(nn.Conv2d(c_in, c_out, k)):>9,}")

# 2) multi-channel convolution: sum the per-channel results, add the bias of that output channel
def conv2d_multichannel_by_hand(x, weight, bias):
    C_out, C_in = weight.shape[:2]
    outs = []
    for o in range(C_out):
        acc = sum(conv2d_by_hand(x[c], weight[o, c]) for c in range(C_in)) + bias[o]
        outs.append(acc)
    return torch.stack(outs)                         # (C_out, H_out, W_out)

x, w, b = torch.randn(2, 5, 5), torch.randn(3, 2, 3, 3), torch.randn(3)
mine = conv2d_multichannel_by_hand(x, w, b)
ref = F.conv2d(x[None], w, b)[0]
print("multi-channel by hand matches F.conv2d:", torch.allclose(mine, ref, atol=1e-5), tuple(mine.shape))

# 3) a conv layer does not care about the image size; a linear layer is welded to it
print("conv1 on 28x28 :", tuple(conv1(torch.rand(1, 1, 28, 28)).shape), " params", count_params(conv1))
print("conv1 on 100x100:", tuple(conv1(torch.rand(1, 1, 100, 100)).shape), " params", count_params(conv1))
lin = nn.Linear(28 * 28, 8)
print("linear on 28x28:", tuple(lin(torch.rand(1, 1, 28, 28).flatten(1)).shape))
try:
    lin(torch.rand(1, 1, 100, 100).flatten(1))
except RuntimeError as e:
    print("linear on 100x100: RuntimeError -", str(e).splitlines()[0][:80])

# 4) weight[3, 5] is the 3x3 slice of output kernel 3 that looks at input channel 5.
#    weight[:, 5] is "how every one of the 16 kernels reads input channel 5": shape (16, 3, 3).
print(tuple(conv2.weight[3, 5].shape), tuple(conv2.weight[:, 5].shape))
```

</details>

---
<a id="s5"></a>
# 5 · The Other Layers

[⬆ back to TOC](#toc)

A CNN is not only convolutions. Here is every other layer type you will meet in a standard image
classifier, each with one demo. Keep the running example in mind: a batch `(N, 1, 28, 28)` of digits.

## 5.1 ReLU — the non-linearity between convolutions

Lab 02 §7.4 showed that stacked *linear* layers collapse into one linear layer. Convolutions are linear
too: two 3 × 3 convolutions in a row **with nothing between them** are exactly one 5 × 5 convolution — the
second layer bought nothing. So every convolution is followed by a non-linearity, almost always **ReLU**:
$\max(0, z)$. On a feature map, ReLU keeps the positive responses and sets everything else to zero.

In [ ]:
before = F.conv2d(digit[None, None], HAND_KERNELS["vertical edges"][None, None], padding=1)[0, 0]
after = F.relu(before)
show_images([digit, before, after], size=2.4, cmap="RdBu_r", vmin=-3, vmax=3,
            titles=["input", "conv output (signed)", "after ReLU (>= 0 only)"])
print(f"before ReLU: min {before.min():.2f}  max {before.max():.2f}   |  after: min {after.min():.2f}  max {after.max():.2f}")

<a id="s52"></a>
## 5.2 Pooling — summarising a neighbourhood

A **pooling** layer slides a window over each feature map like a convolution, but instead of a weighted sum
it takes a fixed summary — the **maximum** (`MaxPool2d`) or the **average** (`AvgPool2d`). Three facts:

* It has **no parameters** — nothing to learn.
* It is applied to **every channel separately**: `(N, C, H, W) → (N, C, H/2, W/2)` for a 2 × 2 window.
* The usual setting `MaxPool2d(kernel_size=2, stride=2)` **halves** height and width; the output-size
  formula of [§3](#s3) applies unchanged.

Max pooling answers "*was this feature present anywhere in this 2 × 2 block?*" — it keeps the strongest
response and forgets exactly where it was. That small forgetting is deliberate: it makes the next layer a
little **tolerant to small shifts**, and it shrinks the computation four-fold.

In [ ]:
grid = torch.tensor([[1., 3., 2., 0.],
                     [4., 2., 1., 1.],
                     [0., 1., 5., 6.],
                     [2., 1., 3., 4.]])

max_pooled = F.max_pool2d(grid[None, None], kernel_size=2, stride=2)[0, 0]
avg_pooled = F.avg_pool2d(grid[None, None], kernel_size=2, stride=2)[0, 0]
show_images([grid, max_pooled, avg_pooled], size=2.3, annotate=True, fmt="{:.2f}", cmap="Blues",
            titles=["4×4 input", "max pool 2×2 -> 2×2", "avg pool 2×2 -> 2×2"])

# on real feature maps: every channel is pooled independently
maps_relu = F.relu(conv1(X_test[:1]))                              # (1, 8, 28, 28)
pooled = F.max_pool2d(maps_relu, 2)                                 # (1, 8, 14, 14)
print("before pooling:", tuple(maps_relu.shape), "  after MaxPool2d(2):", tuple(pooled.shape),
      "  parameters: 0")
show_images(list(maps_relu[0, :4]) + list(pooled[0, :4]), ncols=4, size=1.8, cmap="gray",
            titles=[f"map {i} 28×28" for i in range(4)] + [f"pooled {i} 14×14" for i in range(4)])

### 🧪 Demo — The small-shift tolerance, measured

Shift the digit by one pixel and compare feature maps before and after pooling. The pooled maps change
*relatively* less: a one-pixel move often stays inside the same 2 × 2 window, so the maximum is unchanged.

In [ ]:
def relative_change(a, b):
    """Mean absolute difference between two tensors, relative to their mean magnitude."""
    return ((a - b).abs().mean() / (a.abs().mean() + b.abs().mean()) * 2).item()


x0, x1 = X_test[:1], shift_images(X_test[:1], 1, 0)                 # the same digit, moved one pixel right
with torch.no_grad():
    m0, m1 = F.relu(conv1(x0)), F.relu(conv1(x1))
    p0, p1 = F.max_pool2d(m0, 2), F.max_pool2d(m1, 2)
    pp0, pp1 = F.max_pool2d(p0, 2), F.max_pool2d(p1, 2)

print(f"relative change caused by a 1-pixel shift")
print(f"  conv + ReLU maps (28x28) : {relative_change(m0, m1):.3f}")
print(f"  after one 2x2 max-pool   : {relative_change(p0, p1):.3f}")
print(f"  after two 2x2 max-pools  : {relative_change(pp0, pp1):.3f}")

> 📝 **Pooling or stride?** A `MaxPool2d(2)` after a stride-1 convolution and a single stride-2
> convolution both halve the resolution. Classic networks (LeNet, VGG) pool; many modern ones (ResNet and
> later) use stride and let the convolution learn *how* to downsample. On MNIST both work — you compare them
> in [§9](#s9).

## 5.3 Flatten and the fully-connected head

After a few convolution / pooling stages we have a small stack of feature maps, say `(N, 16, 7, 7)`. To
produce ten class scores we go back to what Lab 02 did: **flatten** to `(N, 16 · 7 · 7) = (N, 784)` and
apply one or two `nn.Linear` layers. Flattening *here* is fine — by now every one of the 784 numbers already
summarises a whole region of the image, so the arrangement has done its job.

In [ ]:
features = F.max_pool2d(F.relu(conv2(F.max_pool2d(F.relu(conv1(X_test[:5])), 2))), 2)
flat = nn.Flatten()(features)
head = nn.Linear(16 * 7 * 7, 10)
print("feature maps :", tuple(features.shape))
print("flattened    :", tuple(flat.shape), "  (16 * 7 * 7 =", 16 * 7 * 7, ")")
print("class scores :", tuple(head(flat).shape))
print("head params  :", f"{count_params(head):,}", "<- the dense head is where most parameters usually sit")

<a id="s54"></a>
## 5.4 Three layers you will see in every modern CNN

| Layer | What it does | Parameters | `train()` vs `eval()` |
|:--|:--|:--|:--|
| `nn.Dropout(p)` | during training, zeroes each input with probability $p$ and scales the rest by $1/(1-p)$; a cheap, strong regulariser | 0 | active only in `train()` |
| `nn.BatchNorm2d(C)` | normalises every channel to mean 0, std 1 over the batch, then re-scales with a learned $\gamma$ and shifts with a learned $\beta$; makes deep nets train faster and more stably | $2C$ | uses batch statistics in `train()`, running averages in `eval()` |
| `nn.AdaptiveAvgPool2d(1)` | averages each feature map to a single number — **global average pooling** — so the head sees `(N, C)` whatever the image size was | 0 | same |

You do not need to master them today. You need to recognise them, know that two of them behave
differently in training and evaluation (that is what `model.train()` / `model.eval()` is for), and be able to
try them in [§9](#s9).

In [ ]:
torch.manual_seed(SEED)
ones = torch.ones(1, 8)

drop = nn.Dropout(p=0.5)
drop.train();  print("Dropout, train(): ", drop(ones).squeeze().tolist(), " <- random zeros, survivors x2")
drop.eval();   print("Dropout, eval() : ", drop(ones).squeeze().tolist(), " <- identity")

bn = nn.BatchNorm2d(8)                                            # one gamma and one beta per channel
normed = bn(conv1(X_test[:64]))
print("\nBatchNorm2d: output", tuple(normed.shape), " params", count_params(bn), "= 2 * 8")
print("   per-channel mean after BN:", [f"{m:.2f}" for m in normed.mean(dim=(0, 2, 3)).tolist()[:4]], "...")
print("   per-channel std  after BN:", [f"{s:.2f}" for s in normed.std(dim=(0, 2, 3)).tolist()[:4]], "...")

gap = nn.AdaptiveAvgPool2d(1)
print("\nglobal average pooling:", tuple(gap(torch.randn(2, 16, 7, 7)).shape),
      " and for a bigger image:", tuple(gap(torch.randn(2, 16, 25, 25)).shape), " params", count_params(gap))

### ✍️ Exercise 5 — The layers, by hand

1. **Max pooling by hand.** Write `maxpool2d_by_hand(x, k=2)` with two loops (like `conv2d_by_hand`) and
   verify it against `F.max_pool2d` on the 4 × 4 `grid` and on the digit.
2. **Average pooling is a convolution.** Show numerically that `F.avg_pool2d(x, 2)` equals
   `F.conv2d(x, w, stride=2)` with a suitable constant 2 × 2 kernel `w`. What is the kernel?
3. **Shape chain.** Predict the output shape after each step of the following stack applied to a
   `(32, 1, 28, 28)` batch, *then* verify by running it one module at a time:
   `Conv2d(1, 6, 5)` → `ReLU` → `MaxPool2d(2)` → `Conv2d(6, 16, 5)` → `ReLU` → `MaxPool2d(2)` → `Flatten` → `Linear(?, 10)`.
   What must `?` be? (This is the shape chain of **LeNet-5**, 1998 — the first successful CNN, built for
   exactly this dataset.)
4. **Dropout keeps the mean.** Pass a batch of ones through `nn.Dropout(0.3)` in training mode 1000 times
   and average the result. What do you get, and why does that make the `eval()` behaviour (identity) the
   right thing to do?

In [ ]:
# ✍️ YOUR CODE HERE
# 1) def maxpool2d_by_hand(x, k=2): ...
# 2) avg_pool2d == conv2d with a constant kernel
# 3) shape chain of LeNet-5
# 4) dropout mean over 1000 draws

<details>
<summary>✅ <b>Show solution</b></summary>

```python
# 1) max pooling by hand
def maxpool2d_by_hand(x, k=2):
    H, W = x.shape
    out = torch.zeros(H // k, W // k)
    for i in range(H // k):
        for j in range(W // k):
            out[i, j] = x[i * k:(i + 1) * k, j * k:(j + 1) * k].max()
    return out

print("grid :", torch.allclose(maxpool2d_by_hand(grid), F.max_pool2d(grid[None, None], 2)[0, 0]))
print("digit:", torch.allclose(maxpool2d_by_hand(digit), F.max_pool2d(digit[None, None], 2)[0, 0]))

# 2) average pooling = convolution with a kernel of 1/4, stride 2, no padding
w = torch.full((1, 1, 2, 2), 0.25)
x = digit[None, None]
print("avg pool == conv(1/4 kernel, stride 2):", torch.allclose(F.avg_pool2d(x, 2), F.conv2d(x, w, stride=2), atol=1e-6))

# 3) LeNet-5 shape chain: 28 -> 24 -> 24 -> 12 -> 8 -> 8 -> 4 -> 16*4*4 = 256 -> 10
layers = [nn.Conv2d(1, 6, 5), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(6, 16, 5), nn.ReLU(), nn.MaxPool2d(2),
          nn.Flatten(), nn.Linear(16 * 4 * 4, 10)]
h = torch.zeros(32, 1, 28, 28)
for layer in layers:
    h = layer(h)
    print(f"{layer.__class__.__name__:<10} -> {tuple(h.shape)}")

# 4) the mean survives: E[dropout(x)] = (1 - p) * x / (1 - p) = x
drop = nn.Dropout(0.3).train()
mean_out = torch.stack([drop(torch.ones(1, 1000)) for _ in range(1000)]).mean()
print(f"mean of dropout(1) over many draws: {mean_out:.3f}  (so eval() = identity keeps the scale)")
```

</details>

---
<a id="s6"></a>
# 6 · Assembling a CNN

[⬆ back to TOC](#toc)

## 6.1 The standard recipe

Almost every image classifier follows the same two-part plan:

1. a **feature extractor** — repeated blocks of `Conv → ReLU → Pool`, each block halving the resolution
   and (usually) doubling the channels, so that spatial detail is traded for a growing vocabulary of
   features;
2. a **classifier head** — `Flatten → Linear → ReLU → Linear`, producing one score per class.

Ours has two blocks. Every shape below comes from the formula in [§3](#s3); check each one as you read.

In [ ]:
cnn_diagram = """
flowchart LR
    I["input<br/>(1, 28, 28)"] --> C1["Conv2d 1→8<br/>k=3, p=1<br/>(8, 28, 28)"]
    C1 --> R1["ReLU"] --> P1["MaxPool 2<br/>(8, 14, 14)"]
    P1 --> C2["Conv2d 8→16<br/>k=3, p=1<br/>(16, 14, 14)"]
    C2 --> R2["ReLU"] --> P2["MaxPool 2<br/>(16, 7, 7)"]
    P2 --> FL["Flatten<br/>(784,)"] --> L1["Linear 784→64"] --> R3["ReLU"] --> L2["Linear 64→10<br/>logits"]
    subgraph feature extractor
        C1; R1; P1; C2; R2; P2
    end
    subgraph classifier head
        FL; L1; R3; L2
    end
"""
render_mermaid(cnn_diagram)

In [ ]:
class SimpleCNN(nn.Module):
    """Two Conv-ReLU-Pool blocks, then a small fully-connected head."""

    def __init__(self, n_hidden=64):
        super().__init__()
        # --- feature extractor -------------------------------------------------------
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)     # (1, 28, 28) -> (8, 28, 28)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)    # (8, 14, 14) -> (16, 14, 14)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)          # halves H and W (used twice)
        self.relu = nn.ReLU()
        # --- classifier head -----------------------------------------------------------
        self.flatten = nn.Flatten()                                # (16, 7, 7) -> (784,)
        self.fc1 = nn.Linear(16 * 7 * 7, n_hidden)
        self.fc2 = nn.Linear(n_hidden, 10)

    def features(self, x):
        """The convolutional part: (N, 1, 28, 28) -> (N, 16, 7, 7)."""
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        return x

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        return self.fc2(x)                                         # logits


torch.manual_seed(SEED)
cnn = SimpleCNN()
print(cnn)
print("\nCNN parameters:", f"{count_params(cnn):,}", "   MLP parameters:", f"{count_params(mlp):,}")

### 🧪 Demo — Tracing the shapes with forward hooks

A **forward hook** is a small function PyTorch calls every time a module produces an output. Attaching one
to every layer prints the shape chain of *any* model — the first thing to do whenever a network gives you a
size error.

In [ ]:
def trace_shapes(model, x):
    """Run x through model and print the output shape and parameter count of every leaf module, in call order."""
    names = {id(m): name for name, m in model.named_modules()}
    handles = []

    def hook(module, inputs, output):
        print(f"{names[id(module)]:<9} {module.__class__.__name__:<10} "
              f"{str(tuple(inputs[0].shape)):<20} -> {str(tuple(output.shape)):<20} params {count_params(module):>7,}")

    for module in model.modules():
        if len(list(module.children())) == 0:                    # leaf modules only
            handles.append(module.register_forward_hook(hook))
    print(f"{'layer':<9} {'type':<10} {'input':<20}    {'output':<20} {'params':>13}")
    print("-" * 82)
    device = next(model.parameters()).device                  # run wherever the model lives
    with torch.no_grad():
        model(x.to(device))
    for h in handles:
        h.remove()


trace_shapes(cnn, torch.zeros(1, 1, 28, 28))

Where do the parameters live?

| Layer | Parameters | Share |
|:--|--:|--:|
| `conv1` (1→8, 3×3) | 80 | 0.2 % |
| `conv2` (8→16, 3×3) | 1 168 | 2.2 % |
| `fc1` (784→64) | 50 240 | 96.3 % |
| `fc2` (64→10) | 650 | 1.2 % |
| **total** | **52 138** | |

The **two convolutions together are 2.4 %** of the model, yet they do all the "seeing". The dense head is
where the parameters are — which is why later architectures replace `Flatten + Linear` with global average
pooling ([§5.4](#s54)). And the whole CNN is **smaller than the MLP** (79 510) that it is about to beat.

## 6.2 The receptive field: how much of the image does one unit see?

A unit in `conv1` sees a 3 × 3 patch. A unit in `conv2` sees a 3 × 3 patch *of pooled conv1 outputs*, and
each of those summarises a 2 × 2 block of 3 × 3 patches — so it sees a **larger** region of the original
image. This region is the unit's **receptive field**; stacking layers is how a CNN goes from edges to
strokes to whole digits.

Track two numbers layer by layer — the receptive field $r$ and the *jump* $j$ (the distance in input pixels
between two neighbouring units):

$$
r_{\text{out}} = r_{\text{in}} + (k - 1)\, j_{\text{in}}, \qquad j_{\text{out}} = j_{\text{in}} \cdot s
$$

| Layer | $k$ | $s$ | $r$ | $j$ |
|:--|:--|:--|:--|:--|
| input | | | 1 | 1 |
| `conv1` | 3 | 1 | 3 | 1 |
| `pool` | 2 | 2 | 4 | 2 |
| `conv2` | 3 | 1 | 8 | 2 |
| `pool` | 2 | 2 | **10** | 4 |

Each of the `16 × 7 × 7` numbers entering the head describes a **10 × 10** patch of the digit. Let us
verify with autograd: the receptive field of an output unit is *the set of input pixels its gradient is
non-zero for*. (We use average pooling and skip ReLU for the measurement, because a max or a ReLU would zero
some gradients and hide part of the field; the geometry is identical.)

In [ ]:
def receptive_field_by_gradient(layers, n=28, channel=0):
    """Mask of input pixels that influence the central output unit of a stack of linear layers."""
    x = torch.zeros(1, 1, n, n, requires_grad=True)
    y = nn.Sequential(*layers)(x)
    c = y.shape[-1] // 2
    y[0, channel, c, c].backward()
    mask = x.grad[0, 0] != 0
    rows, cols = mask.any(dim=1).nonzero().flatten(), mask.any(dim=0).nonzero().flatten()
    return mask, (rows.max() - rows.min() + 1).item(), (cols.max() - cols.min() + 1).item()


torch.manual_seed(SEED)
stages = {
    "conv1":               [nn.Conv2d(1, 8, 3, padding=1)],
    "conv1+pool":          [nn.Conv2d(1, 8, 3, padding=1), nn.AvgPool2d(2)],
    "conv1+pool+conv2":    [nn.Conv2d(1, 8, 3, padding=1), nn.AvgPool2d(2), nn.Conv2d(8, 16, 3, padding=1)],
    "…+pool":              [nn.Conv2d(1, 8, 3, padding=1), nn.AvgPool2d(2), nn.Conv2d(8, 16, 3, padding=1), nn.AvgPool2d(2)],
}
masks, titles = [], []
for name, layers in stages.items():
    mask, h, w = receptive_field_by_gradient(layers)
    print(f"{name:<20} receptive field {h} x {w}")
    masks.append(mask.float()); titles.append(f"{name}: {h}×{w}")
show_images(masks, titles=titles, size=2.3, cmap="gray", suptitle="input pixels seen by the central unit of each stage")

### ✍️ Exercise 6 — Build, trace, count

1. **A third block.** Write `DeeperCNN` with three `Conv(3×3, p=1) → ReLU → MaxPool(2)` blocks with
   8, 16 and 32 channels. Trace its shapes — what is the input size of its first `Linear`? (Careful:
   7 pooled by 2 is 3, not 3.5.) How many parameters does it have compared with `SimpleCNN`?
2. **A head without `Flatten`.** Copy `SimpleCNN` but replace `flatten + fc1` by `nn.AdaptiveAvgPool2d(1)`
   followed by `nn.Flatten()` and a `Linear(16, 10)`. Count the parameters. Then feed it a `(1, 1, 56, 56)`
   image — why does this version work and `SimpleCNN` not?
3. **Receptive field, both ways.** For the LeNet-5 stack of Exercise 5.3 (`Conv 5 → Pool 2 → Conv 5 → Pool 2`),
   compute the receptive field with the $r, j$ table by hand, then verify with `receptive_field_by_gradient`.
4. **Hooks that measure.** Modify `trace_shapes` to also print the *number of output values* of each layer
   (`output.numel()`) — the memory a layer needs during training. Which layer of `SimpleCNN` is the largest
   in memory, and which has the most parameters? They are not the same one.

In [ ]:
# ✍️ YOUR CODE HERE
# 1) class DeeperCNN(nn.Module): ...   then trace_shapes and count_params
# 2) a global-average-pooling head; try a 56x56 input
# 3) LeNet receptive field: by hand, then by gradient
# 4) trace_shapes with output.numel()

<details>
<summary>✅ <b>Show solution</b></summary>

```python
# 1) three blocks: 28 -> 14 -> 7 -> 3, so the head sees 32 * 3 * 3 = 288 numbers
class DeeperCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # -> (8, 14, 14)
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # -> (16, 7, 7)
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # -> (32, 3, 3)
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(32 * 3 * 3, 64), nn.ReLU(), nn.Linear(64, 10))

    def forward(self, x):
        return self.head(self.features(x))

deeper = DeeperCNN()
trace_shapes(deeper, torch.zeros(1, 1, 28, 28))
print("DeeperCNN params:", f"{count_params(deeper):,}", " SimpleCNN:", f"{count_params(SimpleCNN()):,}")
# Deeper, yet FEWER parameters: the extra pooling shrank the flattened vector from 784 to 288.

# 2) global average pooling head: only 16*10 + 10 = 170 head parameters; works at any input size
class GAPCNN(SimpleCNN):
    def __init__(self):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(16, 10)

    def forward(self, x):
        return self.fc(self.flatten(self.gap(self.features(x))))

gapnet = GAPCNN()
print("GAP head, 28x28:", tuple(gapnet(torch.zeros(1, 1, 28, 28)).shape), " 56x56:", tuple(gapnet(torch.zeros(1, 1, 56, 56)).shape))
print("trainable params actually used:", f"{count_params(gapnet.conv1) + count_params(gapnet.conv2) + count_params(gapnet.fc):,}")
# SimpleCNN on 56x56 would flatten to 16*14*14 = 3136 values and crash in fc1 (expects 784).

# 3) LeNet: input r=1,j=1 -> conv5: r=5,j=1 -> pool2: r=6,j=2 -> conv5: r=6+4*2=14,j=2 -> pool2: r=14+1*2=16
mask, h, w = receptive_field_by_gradient([nn.Conv2d(1, 6, 5), nn.AvgPool2d(2), nn.Conv2d(6, 16, 5), nn.AvgPool2d(2)])
print("LeNet receptive field by gradient:", h, "x", w, " (by hand: 16 x 16)")

# 4) activation memory vs parameters
def trace_shapes_mem(model, x):
    names = {id(m): name for name, m in model.named_modules()}
    def hook(module, inputs, output):
        print(f"{names[id(module)]:<9} {module.__class__.__name__:<10} -> {str(tuple(output.shape)):<18} "
              f"values {output.numel():>7,}   params {count_params(module):>7,}")
    handles = [m.register_forward_hook(hook) for m in model.modules() if len(list(m.children())) == 0]
    with torch.no_grad():
        model(x.to(next(model.parameters()).device))
    for h in handles:
        h.remove()

trace_shapes_mem(cnn, torch.zeros(1, 1, 28, 28))
# Largest in memory: conv1's output (8*28*28 = 6272 values per image). Most parameters: fc1.
```

</details>

---
<a id="s7"></a>
# 7 · Training on MNIST

[⬆ back to TOC](#toc)

The training loop is *unchanged* from [§1.3](#s1) — that is the point of `nn.Module`: the loop does not
know or care whether the model flattens first or convolves first. Three epochs are enough to see the
difference. (Roughly 15–40 s per epoch on a CPU, a few seconds on a GPU.)

In [ ]:
torch.manual_seed(SEED)
cnn = SimpleCNN()
history_cnn = train_model(cnn, train_loader, val_loader, epochs=3)

test_loss_cnn, test_acc_cnn = evaluate(cnn, test_loader)
print(f"\nCNN test accuracy: {test_acc_cnn:.4f}     (MLP after 2 epochs: {test_acc_mlp:.4f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
plot_history({"MLP (79,510 params)": history_mlp, "CNN (52,138 params)": history_cnn}, "val_loss", ax=axes[0])
plot_history({"MLP (79,510 params)": history_mlp, "CNN (52,138 params)": history_cnn}, "val_acc", ax=axes[1])
axes[0].set_title("validation loss"); axes[1].set_title("validation accuracy")
plt.tight_layout(); plt.show()

### 🧪 Demo — The two experiments of §1.4, repeated on the CNN

Shift the test digits again. For shifts of one or two pixels the CNN is far more robust than the MLP —
that is the pooling tolerance of [§5.2](#s52) at work. For larger shifts the CNN degrades too: two
max-pools forgive about two pixels, and the dense head is still welded to positions. Robustness to *large*
shifts has to come from the data side (**data augmentation**, Lab 05), not from the architecture alone.

In [ ]:
print(f"{'shift (px)':>10} | {'MLP':>7} | {'CNN':>7}")
print("-" * 30)
shift_results = {"MLP": [], "CNN": []}
for s in range(0, 7):
    loader_s = make_loader(shift_images(X_test, s, s), y_test)
    a_mlp, a_cnn = evaluate(mlp, loader_s)[1], evaluate(cnn, loader_s)[1]
    shift_results["MLP"].append(a_mlp); shift_results["CNN"].append(a_cnn)
    print(f"{s:>10} | {a_mlp:>7.4f} | {a_cnn:>7.4f}")

plt.figure(figsize=(5.2, 3.4))
for name, accs in shift_results.items():
    plt.plot(range(7), accs, marker="o", label=name)
plt.xlabel("shift of the test digits (pixels, right and down)"); plt.ylabel("test accuracy")
plt.grid(alpha=0.3); plt.legend(); plt.show()

### 🧪 Demo — Where does the CNN still fail?

A **confusion matrix** counts, for every true class (rows), what the model predicted (columns). The
diagonal is the correct answers; every off-diagonal cell is a specific *kind* of mistake.

In [ ]:
def predict(model, X, batch_size=1000, device=DEVICE):
    """Predicted class for every image in X."""
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            preds.append(model(X[i:i + batch_size].to(device)).argmax(dim=1).cpu())
    return torch.cat(preds)


y_pred = predict(cnn, X_test)
confusion = torch.bincount(y_test * 10 + y_pred, minlength=100).view(10, 10)   # rows: true, cols: predicted

fig, ax = plt.subplots(figsize=(5.4, 5))
ax.imshow(confusion, cmap="Blues")
for (i, j), v in np.ndenumerate(confusion.numpy()):
    if i != j and v > 0 or i == j:
        ax.text(j, i, str(v), ha="center", va="center", fontsize=7, color="white" if i == j else "black")
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("CNN confusion matrix (test set)")
plt.tight_layout(); plt.show()

off = confusion.clone(); off.fill_diagonal_(0)
top = off.flatten().topk(3).indices
print("most frequent confusions (true -> predicted):",
      [(int(t // 10), int(t % 10), int(off.flatten()[t])) for t in top])

wrong = (y_pred != y_test).nonzero().flatten()
print(f"\n{len(wrong)} test images wrong out of {len(y_test)}")
show_images(X_test[wrong[:16]], ncols=8, size=1.3,
            titles=[f"true {t}, said {p}" for t, p in zip(y_test[wrong[:16]].tolist(), y_pred[wrong[:16]].tolist())])

> 🔑 **Why the CNN wins with fewer parameters.** The MLP had to learn "a stroke here" and "the same stroke
> two pixels to the right" as two unrelated facts, with separate weights. The CNN learns the stroke *once*
> (weight sharing), detects it *everywhere* (equivariance), and pooling lets it not care about a pixel or two
> of position. That is prior knowledge about images — locality and translation — **built into the
> architecture** instead of being learned from data. Fewer parameters, better accuracy, and much better
> behaviour on inputs it did not see.

### ✍️ Exercise 7 — Training experiments

1. **Shuffled pixels, again.** Train a fresh `SimpleCNN` on the *pixel-shuffled* images of [§1.4](#s14)
   (`shuffle_pixels(X_train)` etc.) for 2 epochs. In §1.4 the MLP did not care. Does the CNN? Explain the
   result in one sentence using the word *locality*.
2. **Longer training.** Train `SimpleCNN` for 8 epochs and plot training vs validation accuracy. Do the
   curves separate? Which one keeps rising?
3. **Optimiser.** Replace Adam by plain SGD (`torch.optim.SGD(..., lr=0.05)`) inside a copy of
   `train_model` (or add an `optimizer_class` argument). Compare validation accuracy after 3 epochs.
4. **Per-class accuracy.** From `confusion`, compute the accuracy of each digit. Which digit is hardest for
   the CNN? Was it the same one for the MLP (`predict(mlp, X_test)`)?

In [ ]:
# ✍️ YOUR CODE HERE
# 1) SimpleCNN on shuffle_pixels(X_train), shuffle_pixels(X_val); evaluate on shuffle_pixels(X_test)
# 2) 8 epochs; plot train_acc vs val_acc
# 3) SGD vs Adam
# 4) per-class accuracy from the confusion matrix, for CNN and MLP

<details>
<summary>✅ <b>Show solution</b></summary>

```python
# 1) the CNN DOES care: its kernels assume that neighbouring inputs are related (locality).
#    Shuffling destroys that, and the CNN now does WORSE than the MLP on the same shuffled images.
torch.manual_seed(SEED)
cnn_shuffled = SimpleCNN()
train_model(cnn_shuffled, make_loader(shuffle_pixels(X_train), y_train, shuffle=True),
            make_loader(shuffle_pixels(X_val), y_val), epochs=2)
print("CNN on shuffled pixels:", f"{evaluate(cnn_shuffled, make_loader(shuffle_pixels(X_test), y_test))[1]:.4f}",
      "   CNN on real images:", f"{test_acc_cnn:.4f}")

# 2) longer training: training accuracy keeps rising towards 100 %, validation flattens
torch.manual_seed(SEED)
cnn_long = SimpleCNN()
h = train_model(cnn_long, train_loader, val_loader, epochs=8, verbose=False)
plot_history({"train": {"acc": h["train_acc"]}, "validation": {"acc": h["val_acc"]}}, "acc")
plt.show()

# 3) SGD: slower to start at this learning rate; Adam reaches a higher accuracy in 3 epochs
def train_sgd(model, epochs=3, lr=0.05):
    model.to(DEVICE)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss = F.cross_entropy(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
        print(f"epoch {epoch + 1}: val acc {evaluate(model, val_loader)[1]:.4f}")

torch.manual_seed(SEED)
train_sgd(SimpleCNN())

# 4) per-class accuracy
per_class_cnn = confusion.diag() / confusion.sum(dim=1)
conf_mlp = torch.bincount(y_test * 10 + predict(mlp, X_test), minlength=100).view(10, 10)
per_class_mlp = conf_mlp.diag() / conf_mlp.sum(dim=1)
for c in range(10):
    print(f"digit {c}: CNN {per_class_cnn[c]:.3f}   MLP {per_class_mlp[c]:.3f}")
print("hardest for CNN:", per_class_cnn.argmin().item(), "  hardest for MLP:", per_class_mlp.argmin().item())
```

</details>

---
<a id="s8"></a>
# 8 · Looking Inside

[⬆ back to TOC](#toc)

A CNN is unusually inspectable: its first-layer kernels are tiny images we can draw, and every intermediate
result is a stack of pictures. Three views into the trained `cnn`.

### 🧪 Demo — The learned first-layer kernels

Compare with the random kernels of [§4](#s4) and the hand-designed ones of [§2.2](#s2). Training has
turned noise into oriented edge detectors and centre–surround patterns — the same shapes engineers used to
draw by hand.

In [ ]:
cnn.eval()
learned = cnn.conv1.weight[:, 0]                       # show_images copies tensors to the CPU for plotting
show_images(learned, titles=[f"kernel {i}" for i in range(8)], size=1.5, cmap="RdBu_r",
            annotate=True, fmt="{:.1f}", suptitle="conv1 kernels after training")

### 🧪 Demo — One digit, layer by layer

Follow a single test image through the feature extractor. Notice how the maps become **coarser** (28 → 14 → 7)
and **more abstract**: conv1 maps still look like the digit; conv2 maps respond to *combinations* of
strokes in a 10 × 10 region and are no longer pictures a human reads at a glance.

In [ ]:
x = X_test[:1].to(DEVICE)                             # the 7, on the model's device
with torch.no_grad():
    a1 = cnn.relu(cnn.conv1(x))                       # (1, 8, 28, 28)
    p1 = cnn.pool(a1)                                 # (1, 8, 14, 14)
    a2 = cnn.relu(cnn.conv2(p1))                      # (1, 16, 14, 14)
    p2 = cnn.pool(a2)                                 # (1, 16, 7, 7)

show_images([x[0, 0]], titles=["input (1, 28, 28)"], size=2)
show_images(a1[0], titles=[f"conv1+ReLU {i}" for i in range(8)], size=1.5, suptitle="after conv1 + ReLU: (8, 28, 28)")
show_images(p1[0], titles=[f"pool {i}" for i in range(8)], size=1.5, suptitle="after max-pool: (8, 14, 14)")
show_images(a2[0], titles=[f"conv2+ReLU {i}" for i in range(16)], size=1.5, suptitle="after conv2 + ReLU: (16, 14, 14)")
show_images(p2[0], titles=[f"pool {i}" for i in range(16)], size=1.5, suptitle="after max-pool: (16, 7, 7) - what the head sees")

### 🧪 Demo — What does each conv2 channel like?

A feature map we cannot read directly can still be understood by asking: *which test images make this
channel fire most?* Average each conv2 map over its 7 × 7 positions, and show the eight strongest images per
channel. Many channels turn out to prefer a particular digit or stroke.

In [ ]:
with torch.no_grad():
    activations = torch.cat([cnn.features(X_test[i:i + 1000].to(DEVICE)).mean(dim=(2, 3)).cpu()
                             for i in range(0, 10000, 1000)])
print("mean activation per image and channel:", tuple(activations.shape))

for ch in range(4):
    top = activations[:, ch].topk(8).indices
    show_images(X_test[top], titles=[f"label {y}" for y in y_test[top].tolist()], size=1.25,
                suptitle=f"conv2 channel {ch}: the 8 test images that activate it most")

### ✍️ Exercise 8 — Probe the trained network

1. **All sixteen.** Extend the last demo to all 16 channels (4 images each is enough). Roughly how many
   channels are "digit detectors" (one dominant label) and how many are "stroke detectors" (mixed labels)?
2. **Ablation.** For each of the 8 `conv1` kernels in turn, set its weights to zero, measure test accuracy,
   and restore it (`.clone()` first!). Which kernel matters most? Which barely matters — and what does that
   say about redundancy in the layer?
3. **A mirrored digit.** Run the layer-by-layer demo on `torch.flip(x, dims=[3])`. Which conv1 maps swap
   roles with each other, and why is that expected from the kernels you drew?
4. **Dead channels.** Compute, over the whole test set, the fraction of positions where each conv2 map (after
   ReLU) is exactly zero. Is any channel dead (never active)? Dead ReLU units are a real failure mode in
   deeper networks.

In [ ]:
# ✍️ YOUR CODE HERE
# 1) top-4 images for each of the 16 channels
# 2) ablation loop over cnn.conv1.weight.data[k]
# 3) layer-by-layer on torch.flip(x, dims=[3])
# 4) fraction of zeros per conv2 channel

<details>
<summary>✅ <b>Show solution</b></summary>

```python
# 1) all channels
for ch in range(16):
    top = activations[:, ch].topk(4).indices
    print(f"channel {ch:>2}: top labels {y_test[top].tolist()}")

# 2) ablation: zero one kernel at a time
base = evaluate(cnn, test_loader)[1]
print(f"all kernels: {base:.4f}")
for k in range(8):
    saved = cnn.conv1.weight.data[k].clone()
    cnn.conv1.weight.data[k].zero_()
    acc = evaluate(cnn, test_loader)[1]
    cnn.conv1.weight.data[k] = saved
    print(f"without kernel {k}: {acc:.4f}   (drop {base - acc:+.4f})")
# Typically the worst single kernel costs one or two percent and some cost almost nothing - the layer is
# redundant, which is one reason dropout is rarely applied to convolutional layers.

# 3) mirrored input: a left-edge detector becomes a right-edge detector, so maps with mirrored kernels swap
xf = torch.flip(x, dims=[3])
with torch.no_grad():
    a1f = cnn.relu(cnn.conv1(xf))
show_images(list(a1[0]) + list(a1f[0]), ncols=8, size=1.4,
            titles=[f"orig {i}" for i in range(8)] + [f"flipped {i}" for i in range(8)])

# 4) dead channels: fraction of (image, row, column) positions where each conv2 map is exactly 0
zeros, total = torch.zeros(16), 0
with torch.no_grad():
    for i in range(0, 10000, 1000):
        a2 = cnn.relu(cnn.conv2(cnn.pool(cnn.relu(cnn.conv1(X_test[i:i + 1000].to(DEVICE))))))   # (1000, 16, 14, 14)
        zeros += (a2 == 0).float().sum(dim=(0, 2, 3)).cpu()
        total += a2[:, 0].numel()
zero_frac = zeros / total
for ch, f in enumerate(zero_frac.tolist()):
    print(f"conv2 channel {ch:>2}: zero at {f:.1%} of positions{'   <- dead?' if f > 0.99 else ''}")
```

</details>

---
<a id="s9"></a>
# 9 · Experiments: Does the Choice Matter?

[⬆ back to TOC](#toc)

Every hyperparameter of this lab, in one factory function. `make_cnn` builds a two-block CNN from a handful
of choices and uses `conv_out_size` to work out the size of the flattened vector — so the head always fits.
`run_experiment` trains it for a fixed budget and records **validation** accuracy, parameters and time in a
shared table. (We select on validation, never on test — the test set stays sealed until the end.)

In [ ]:
def make_cnn(kernel_size=3, n_filters=(8, 16), padding="same", downsample="pool",
             dropout=0.0, batchnorm=False, n_hidden=64):
    """
    Build a two-block CNN for 28x28 grey images.

    Args:
        kernel_size: Side of every convolution kernel (odd).
        n_filters: Output channels of the two convolution blocks.
        padding: "same" -> p = (k-1)//2, or "valid" -> p = 0.
        downsample: "pool" -> stride-1 conv followed by MaxPool2d(2); "stride" -> stride-2 conv, no pooling.
        dropout: Dropout probability in the head (0 disables it).
        batchnorm: Insert BatchNorm2d after every convolution.
        n_hidden: Width of the hidden fully-connected layer.
    """
    layers, c_in, n = [], 1, 28
    for c_out in n_filters:
        p = (kernel_size - 1) // 2 if padding == "same" else 0
        s = 2 if downsample == "stride" else 1
        layers.append(nn.Conv2d(c_in, c_out, kernel_size, stride=s, padding=p))
        n = conv_out_size(n, kernel_size, p, s)
        if batchnorm:
            layers.append(nn.BatchNorm2d(c_out))
        layers.append(nn.ReLU())
        if downsample == "pool":
            layers.append(nn.MaxPool2d(2))
            n = conv_out_size(n, 2, 0, 2)
        c_in = c_out
    layers += [nn.Flatten(), nn.Dropout(dropout), nn.Linear(c_in * n * n, n_hidden), nn.ReLU(),
               nn.Dropout(dropout), nn.Linear(n_hidden, 10)]
    return nn.Sequential(*layers)


RESULTS = {}


def run_experiment(name, model, epochs=2, train_loader=train_loader, val_loader=val_loader, seed=SEED):
    """Train a model for a fixed budget and record validation accuracy, parameter count and time."""
    torch.manual_seed(seed)
    for m in model.modules():                                   # re-initialise for a fair comparison
        if hasattr(m, "reset_parameters"):
            m.reset_parameters()
    t0 = time.time()
    history = train_model(model, train_loader, val_loader, epochs=epochs, verbose=False)
    RESULTS[name] = {"val_acc": history["val_acc"][-1], "params": count_params(model),
                     "seconds": time.time() - t0}
    print(f"{name:<32} val acc {RESULTS[name]['val_acc']:.4f}   params {RESULTS[name]['params']:>8,}   "
          f"{RESULTS[name]['seconds']:5.1f}s")
    return history


def results_table():
    """Print every experiment so far, best validation accuracy first."""
    print(f"{'experiment':<32} {'val acc':>8} {'params':>10} {'seconds':>8}")
    print("-" * 62)
    for name, r in sorted(RESULTS.items(), key=lambda kv: -kv[1]["val_acc"]):
        print(f"{name:<32} {r['val_acc']:>8.4f} {r['params']:>10,} {r['seconds']:>8.1f}")


# the baseline of section 6, rebuilt by the factory, plus one variation to show the pattern
trace_shapes(make_cnn(), torch.zeros(1, 1, 28, 28))
print()
_ = run_experiment("baseline k=3 same pool", make_cnn())
_ = run_experiment("k=5 same pool", make_cnn(kernel_size=5))

### ✍️ Exercise 9 — Fill the table

Run each variation below for **2 epochs** (each one takes as long as the baseline) and add it to the
table with `results_table()`. Before every run, **predict**: more or fewer parameters than the baseline?
Better or worse accuracy? Then explain any surprise.

1. **Kernel size:** `kernel_size=7`.
2. **No padding:** `padding="valid"` (with `k=3` and with `k=5`). Look at the parameter count before the accuracy.
3. **Stride instead of pooling:** `downsample="stride"` — with `kernel_size=3` and again with `kernel_size=5`.
4. **Width:** `n_filters=(4, 8)` and `n_filters=(32, 64)`.
5. **Regularisation:** `dropout=0.5`; `batchnorm=True`; both.
6. **Little data.** Build a loader from the first 1 000 training images
   (`make_loader(X_train[:1000], y_train[:1000], shuffle=True)`) and train the baseline **and** the
   `dropout=0.5` version for 60 epochs each (that is still only 480 updates). Compare *training* and
   *validation* accuracy of the two — which one generalises better, and what happened to the gap between
   the two curves?
7. **Final answer.** Pick the best configuration by *validation* accuracy, retrain it once for 5 epochs, and
   report its **test** accuracy — the only time the test set is used for a decision you made.

In [ ]:
# ✍️ YOUR CODE HERE
# run_experiment("k=7 same pool", make_cnn(kernel_size=7))
# ...
# results_table()

<details>
<summary>✅ <b>Show solution</b></summary>

```python
run_experiment("k=7 same pool", make_cnn(kernel_size=7))
run_experiment("k=3 valid pool", make_cnn(padding="valid"))
run_experiment("k=5 valid pool", make_cnn(kernel_size=5, padding="valid"))
run_experiment("k=3 same stride-2", make_cnn(downsample="stride"))
run_experiment("k=5 same stride-2", make_cnn(kernel_size=5, downsample="stride"))
run_experiment("filters (4, 8)", make_cnn(n_filters=(4, 8)))
run_experiment("filters (32, 64)", make_cnn(n_filters=(32, 64)))
run_experiment("dropout 0.5", make_cnn(dropout=0.5))
run_experiment("batchnorm", make_cnn(batchnorm=True))
run_experiment("dropout 0.5 + batchnorm", make_cnn(dropout=0.5, batchnorm=True))
results_table()
```

What you should see (numbers vary a little between runs):

* **Kernel size** — 5 and 7 cost more parameters in the convolutions (`25` or `49` weights per slice) but
  the *head* dominates the count anyway; accuracy changes little on 28 × 28 digits. Bigger kernels are not
  automatically better; two stacked 3 × 3 layers see 5 × 5 with fewer parameters.
* **Valid padding** — the maps shrink (28 → 26 → 13 → 11 → 5 for `k=3`), so the flattened vector and the
  head get *smaller*: fewer parameters, sometimes *slightly lower* accuracy; border strokes are seen less.
* **Stride 2** — faster (each convolution runs on a quarter of the positions), but with `k=3` it is a
  couple of points *worse* after two epochs: a 3 × 3 kernel jumping 2 pixels covers every input pixel only
  once, so the sparse sampling loses information that conv-then-pool kept. With `k=5` the windows overlap
  again and stride-2 matches pooling. This is why strided networks use overlapping windows.
* **Width** — `(4, 8)` loses a little; `(32, 64)` gains a little at several times the parameters and
  time. Diminishing returns on MNIST.
* **Dropout / batch norm** — on 55 000 examples for 2 epochs there is little overfitting to prevent, so
  dropout may even lower the 2-epoch accuracy (the network trains more slowly); batch norm speeds
  training up and typically gives the best 2-epoch number.

```python
# 6) little data: regularisation earns its keep when data is scarce - but only once the network has had
#    enough updates to over-fit. In our run, after 60 epochs the baseline reached ~100 % training accuracy
#    and ~92 % validation; with dropout, training accuracy stayed lower (~98 %) and validation rose to ~94 %.
#    With only 15 epochs dropout does NOT help yet: the network is still under-fitting, and dropout slows it.
small_train = make_loader(X_train[:1000], y_train[:1000], shuffle=True)
h_base = run_experiment("1k images, baseline", make_cnn(), epochs=60, train_loader=small_train)
h_drop = run_experiment("1k images, dropout 0.5", make_cnn(dropout=0.5), epochs=60, train_loader=small_train)
print(f"train acc after 60 epochs: baseline {h_base['train_acc'][-1]:.3f}   dropout {h_drop['train_acc'][-1]:.3f}")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
plot_history({"baseline": h_base, "dropout 0.5": h_drop}, "train_acc", ax=axes[0]); axes[0].set_title("training accuracy")
plot_history({"baseline": h_base, "dropout 0.5": h_drop}, "val_acc", ax=axes[1]); axes[1].set_title("validation accuracy")
plt.tight_layout(); plt.show()
results_table()

# 7) final model: best by validation, evaluated on test ONCE
best = make_cnn(batchnorm=True)
train_model(best, train_loader, val_loader, epochs=5)
print("final TEST accuracy:", f"{evaluate(best, test_loader)[1]:.4f}")
```

</details>

---
<a id="s10"></a>
# 10 · Self-Check Quiz

[⬆ back to TOC](#toc)

Answer **without scrolling back up**. Aim for 8/10 before Lab 05. Every answer comes with an explanation
and the section to revisit.

In [ ]:
QUESTIONS = [
    {"q": "A 3x3 convolution with stride 1 and NO padding is applied to a 28x28 image. Output size?",
     "options": ["28 x 28", "27 x 27", "26 x 26", "14 x 14"],
     "answer": 2,
     "why": "(28 + 0 - 3) / 1 + 1 = 26. The kernel centre cannot reach the outermost pixel ring. (SS3)"},

    {"q": "How many parameters does nn.Conv2d(8, 16, kernel_size=3) have?",
     "options": ["144", "1152", "1168", "It depends on the image size"],
     "answer": 2,
     "why": "16 kernels, each an (8, 3, 3) block = 1152 weights, plus 16 biases = 1168. The image size never enters. (SS4)"},

    {"q": "Why is padding p = (k - 1) / 2 called 'same' padding?",
     "options": ["The same zeros are used on every side",
                 "With stride 1 the output has the same height and width as the input",
                 "It gives the same result as no padding",
                 "All kernels share the same weights"],
     "answer": 1,
     "why": "(n + 2p - k) + 1 = n exactly when 2p = k - 1. (SS3.2)"},

    {"q": "What does stride 2 do to a feature map, compared with stride 1?",
     "options": ["Doubles the number of channels",
                 "Roughly halves height and width, because the kernel jumps two pixels per step",
                 "Doubles height and width",
                 "Nothing - stride only changes speed"],
     "answer": 1,
     "why": "The kernel visits every second position in each direction: about a quarter of the outputs. (SS3.3)"},

    {"q": "How many trainable parameters does nn.MaxPool2d(2) have?",
     "options": ["4", "2 per channel", "0", "As many as the number of channels"],
     "answer": 2,
     "why": "Pooling is a fixed summary (the maximum); there is nothing to learn. (SS5.2)"},

    {"q": "'Weight sharing' in a convolution layer means…",
     "options": ["All kernels in a layer have identical weights",
                 "The same kernel weights are applied at every spatial position of the input",
                 "Two layers share one weight tensor",
                 "The weights are shared between training and test time"],
     "answer": 1,
     "why": "One detector, reused everywhere. It is what makes the layer translation-equivariant and small. (SS2.3)"},

    {"q": "Why did the MLP collapse on shifted digits while the CNN degraded much less?",
     "options": ["The CNN has more parameters",
                 "The CNN was trained for more epochs",
                 "Convolutions detect a pattern wherever it appears, and pooling adds tolerance to small shifts; the MLP ties every weight to one fixed pixel position",
                 "The MLP used the wrong loss"],
     "answer": 2,
     "why": "Locality + weight sharing = equivariance; pooling = a little invariance. The MLP has neither. (SS1.4, SS7)"},

    {"q": "Two 3x3 convolutions are stacked with NO non-linearity between them. What have you built?",
     "options": ["A more powerful two-layer detector",
                 "Exactly one 5x5 convolution - the second layer added nothing new",
                 "An invalid network - PyTorch will raise an error",
                 "A pooling layer"],
     "answer": 1,
     "why": "Convolutions are linear; linear followed by linear is linear. ReLU between them is not optional. (SS5.1)"},

    {"q": "nn.Conv2d(3, 32, kernel_size=5) is applied to a batch of shape (16, 3, 100, 100). Output shape?",
     "options": ["(16, 3, 96, 96)", "(16, 32, 100, 100)", "(16, 32, 96, 96)", "(16, 32, 20, 20)"],
     "answer": 2,
     "why": "Channels become out_channels = 32; spatial size is (100 - 5) + 1 = 96 without padding. (SS3, SS4)"},

    {"q": "In a CNN with two conv blocks and a dense head, where do most of the parameters usually sit?",
     "options": ["In the first convolution", "In the pooling layers",
                 "In the first fully-connected layer after Flatten", "Spread evenly"],
     "answer": 2,
     "why": "fc1 was 96 % of SimpleCNN. Convolutions are cheap; the flattened vector times the hidden width is not. (SS6.1)"},
]


def grade(answers):
    """Score a list of chosen option indices and explain every question."""
    correct = 0
    print("=" * 78)
    for number, (item, given) in enumerate(zip(QUESTIONS, answers), start=1):
        is_right = given == item["answer"]
        correct += int(is_right)
        print(f"Q{number}. {'CORRECT' if is_right else 'WRONG'}  -  {item['q']}")
        print(f"      you said  : {item['options'][given]}")
        if not is_right:
            print(f"      correct   : {item['options'][item['answer']]}")
        print(f"      why       : {item['why']}\n")
    print("=" * 78)
    print(f"SCORE: {correct}/{len(QUESTIONS)}")
    print("Excellent - you are ready for Lab 05." if correct >= 8 else
          "Good start - revisit the sections named above, then retake the quiz.")
    return correct


try:
    import ipywidgets as widgets
    from IPython.display import display

    pickers, rows = [], []
    for number, item in enumerate(QUESTIONS, start=1):
        picker = widgets.RadioButtons(
            options=[(text, index) for index, text in enumerate(item["options"])],
            value=None, layout=widgets.Layout(width="95%"))
        pickers.append(picker)
        rows.append(widgets.VBox([widgets.HTML(f"<b>Q{number}. {item['q']}</b>"), picker]))

    button = widgets.Button(description="Check my answers", button_style="success", icon="check")
    output = widgets.Output()

    def on_click(_):
        output.clear_output()
        with output:
            unanswered = [i + 1 for i, p in enumerate(pickers) if p.value is None]
            if unanswered:
                print("Please answer every question. Still missing:", unanswered)
                return
            grade([p.value for p in pickers])

    button.on_click(on_click)
    display(widgets.VBox(rows + [button, output]))
except ImportError:
    print("ipywidgets is not available - answer on paper, then put your ten choices in a list")
    print("(one option index per question) and run:")
    print("    grade([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])   # <- replace with YOUR answers\n")
    for number, item in enumerate(QUESTIONS, start=1):
        print(f"Q{number}. {item['q']}")
        for index, option in enumerate(item["options"]):
            print(f"    {index}) {option}")
        print()

---
<a id="s11"></a>
# 11 · Summary, Cheat Sheet & Next Steps

[⬆ back to TOC](#toc)

In [ ]:
summary_diagram = """
flowchart TD
    N["one neuron<br/>z = wᵀx + b"] -->|"look at a k×k patch only"| L["local neuron"]
    L -->|"reuse the same weights<br/>at every position"| C["convolution kernel<br/>k² + 1 parameters"]
    C -->|"k, padding, stride"| S["output size<br/>⌊(n + 2p − k)/s⌋ + 1"]
    C -->|"many kernels"| CH["channels<br/>weight (out, in, k, k)"]
    CH -->|"+ ReLU"| NL["non-linear feature maps"]
    NL -->|"MaxPool 2"| P["coarser, shift-tolerant"]
    P -->|"repeat"| RF["growing receptive field"]
    RF -->|"Flatten + Linear"| H["class scores"]
    H --> T["same loss, same loop<br/>as Lab 02 / 03"]
"""
render_mermaid(summary_diagram)

## 📋 One-page cheat sheet

| Task | Code |
|:--|:--|
| image batch layout | `(N, C, H, W)` — batch, channels, height, width |
| one grey image → batch of one | `img[None, None]` or `img.unsqueeze(0).unsqueeze(0)` |
| a convolution layer | `nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0)` |
| its weight and bias shapes | `(out, in, k, k)` and `(out,)` |
| its parameter count | `out * in * k * k + out` — **independent of H, W** |
| output size (per axis) | `(n + 2p - k) // s + 1` — with dilation: `k -> d*(k-1) + 1` |
| "same" padding (stride 1) | `padding=(k - 1) // 2` or `padding="same"` |
| halve the resolution | `nn.MaxPool2d(2)` **or** a stride-2 convolution |
| functional versions | `F.conv2d(x, w, b, stride, padding)`, `F.max_pool2d(x, 2)`, `F.relu(x)` |
| feature maps → vector | `nn.Flatten()`: `(N, C, H, W) -> (N, C*H*W)` |
| head that ignores image size | `nn.AdaptiveAvgPool2d(1)` then `nn.Flatten()` then `nn.Linear(C, 10)` |
| receptive field, layer by layer | `r_out = r_in + (k - 1) * j_in`, `j_out = j_in * s` |
| receptive field, by autograd | backprop one output unit; look where `x.grad != 0` |
| trace every shape | a `register_forward_hook` on each leaf module |
| train / eval behaviour | `model.train()` before training, `model.eval()` + `torch.no_grad()` before evaluating |
| regularise | `nn.Dropout(p)` in the head, `nn.BatchNorm2d(C)` after a conv |
| verify anything by hand | write the loops, then `torch.allclose(mine, F.something(...))` |

## 🧠 The eight sentences worth memorising

1. A convolution is **one small neuron slid over the image**, using the **same weights** at every position.
2. Its output is a **feature map**: where in the image the kernel's pattern occurs.
3. **Kernel size** sets what one output can see; **padding** keeps the borders and the size; **stride**
   downsamples. One formula gives the output size — $\lfloor (n + 2p - k)/s \rfloor + 1$.
4. A layer has **many kernels**, one **feature map** each; the next layer's kernels are **3-D blocks** that
   read all of them. Parameters: $C_{\text{out}} \cdot C_{\text{in}} \cdot k^2 + C_{\text{out}}$ — never the image size.
5. **ReLU** between convolutions is not optional: two linear convolutions are one convolution.
6. **Pooling** has no parameters; it halves the resolution and forgives a pixel or two of shift.
7. Stacking blocks **grows the receptive field**: edges → strokes → digits.
8. The MLP fails on shifted and does not notice shuffled pixels; the CNN is the reverse — because
   **locality and translation are built into the architecture**, not learned.

## 🔭 What comes next

| Next | Why | Where |
|:--|:--|:--|
| **Data augmentation** | the shift test of §7, fixed from the *data* side: train on shifted, rotated, scaled digits | Lab 05 |
| **Deeper networks: VGG, ResNet, skip connections** | many more blocks than two — and why depth needs help to train | Lecture 05 |
| **Transfer learning** | reuse the feature extractor of a network trained on a million photos | Lab 05/06 |
| **Colour and larger images** | `in_channels=3`, 224 × 224 — same layers, same formula | Lab 05 |
| **Beyond classification** | detection and segmentation reuse the same feature maps, with dilation and upsampling | later |

> 🎓 **The one thing to carry forward.** Every architecture you will meet from now on — VGG, ResNet,
> U-Net, even the patch embedding of a Vision Transformer — is made of the six pieces of this lab: a
> convolution with a kernel size, padding and stride; channels; a non-linearity; pooling or striding; a
> receptive field that grows with depth; and a head. When a network's shapes do not line up, the formula of
> §3 tells you where. When it does not learn, the loop of §1.3 has not changed since Lab 02.

---

*DL2026 · Lab 04 · Convolutional Neural Networks — reference: Raschka, Liu & Mirjalili,
**Machine Learning with PyTorch and Scikit-Learn**, chapter 14.*